In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
from functools import reduce

# 1. LOAD & COMBINE TRIP FILES
trip_dir = "./bikeshare_data/tracts_with_trips"
trip_frames = []
for path in glob.glob(os.path.join(trip_dir, "*_tracts_with_trips.csv")):
    city = os.path.basename(path).split("_")[0]
    df = pd.read_csv(path, dtype={"GEOID": str})
    df = df[["GEOID", "ALAND", "trips_per_day", "total_trips"]].copy()
    df["city"] = city
    trip_frames.append(df)

trips = pd.concat(trip_frames, ignore_index=True)
trips = trips.rename(columns={"GEOID": "geoid", "ALAND": "aland"})

# 2. LOAD & MERGE CENSUS TABLES (DROP DUPLICATE “DECORATIVE” COLS)
census_files = [
    "./census_data/2010_data/S0101_2010.csv",
    "./census_data/2010_data/S0601_2010.csv",
    "./census_data/2010_data/S0701_2010.csv",
    "./census_data/2010_data/S0801_2010.csv",
    "./census_data/2010_data/S0802_2010.csv",
    "./census_data/2010_data/DP03_2010.csv",
]
decorative = {"NAME", "state", "county", "county_name", "TRACTCE", "COUNTYFP", "STATEFP"}
seen_decor = set()
seen_file = set()
census_frames = []

for path in census_files:
    if path in seen_file:
        continue
    seen_file.add(path)

    dfc = pd.read_csv(path, dtype={"GEOID": str})
    dfc = dfc.rename(columns={"GEOID": "geoid", "GEO_ID": "geoid"})
    # Only keep last 11 chars of geoid to align with geoids from tract trips
    dfc["geoid"] = dfc["geoid"].str[-11:]

    # Only keep columns that end in E, eg are estimates rather than margins of error
    keep_cols = ['geoid'] + [c for c in dfc.columns if re.search(r'E$', c)]
    dfc = dfc[keep_cols]

    # drop any decorative cols once we’ve kept them once
    drop = [c for c in dfc.columns if c in decorative and c in seen_decor]
    if drop:
        dfc = dfc.drop(columns=drop)

    seen_decor.update(c for c in dfc.columns if c in decorative)
    census_frames.append(dfc)

census = reduce(
    lambda left, right: pd.merge(left, right, on="geoid", how="outer"),
    census_frames
)

df = pd.merge(trips, census, on="geoid", how="inner")
df = df.dropna(subset=["trips_per_day"])  # make sure target exists
df.to_csv('./multiple_regression/super_linreg_df_2010.csv')


In [ ]:
df.shape

In [ ]:
# ------------------------------------------------------------
# 0.  SET-UP & TYPE COERCION
# ------------------------------------------------------------
import pandas as pd
import numpy as np
import os

# Paths
acs10_path = "./multiple_regression/super_linreg_df_2010.csv"
cross_txt  = "./crosswalks/tab20_tract20_tract10_natl.txt"
out_path   = "./multiple_regression/super_linreg_df_2020.csv"

# ------------------------------------------------------------
# 1. LOAD & COERCE 2010 ACS DATA
# ------------------------------------------------------------
acs10 = pd.read_csv(acs10_path, dtype={'geoid': str})

# Coerce all columns except identifiers to numeric (errors → NaN)
id_cols = ['geoid', 'city', 'NAME']
value_cols = [c for c in acs10.columns if c not in id_cols]
acs10[value_cols] = acs10[value_cols].apply(pd.to_numeric, errors='coerce')

# Keep only the 11-char tract code
acs10['geoid_2010'] = acs10['geoid'].str[-11:]

# ------------------------------------------------------------
# 2.  LOAD CROSSWALK (pipe-delimited TNT)
# ------------------------------------------------------------
xwalk = pd.read_csv(cross_txt, sep='|', dtype=str)
xwalk[['AREALAND_PART', 'AREALAND_TRACT_10']] = xwalk[['AREALAND_PART', 'AREALAND_TRACT_10']].astype(float)
xwalk['pop_w'] = xwalk['AREALAND_PART'] / xwalk['AREALAND_TRACT_10']
xwalk = (
    xwalk.rename(columns={
        'GEOID_TRACT_10':'geoid_2010',
        'GEOID_TRACT_20':'geoid_2020'
    })[['geoid_2010','geoid_2020','pop_w']]
)

# ------------------------------------------------------------
# 3.  MERGE & WEIGHT – FROM 2010 → 2020 TRACTS
# ------------------------------------------------------------
merged = acs10.merge(xwalk, on='geoid_2010', how='inner')

# Now identify numeric columns (after coercion)
exclude = ['aland', 'trips_per_day', 'total_trips', 'pop_w', 
           'geoid_2010', 'geoid', 'city', 'NAME']
numcols = [c for c in merged.columns 
           if c not in exclude and pd.api.types.is_numeric_dtype(merged[c])]

# Multiply each estimate by the area weight
for col in numcols:
    merged[col] = merged[col] * merged['pop_w']

# Aggregate weighted values to 2020 GEOIDs
acs20 = (
    merged
    .groupby('geoid_2020', as_index=False)
    .agg({**{c:'sum' for c in numcols},
          'aland':'first',
          'city':'first',
          'trips_per_day':'first',
          'total_trips':'first'})
)

# Finalize and write out
acs20 = acs20.rename(columns={'geoid_2020':'geoid'})
acs20.to_csv(out_path, index=False)

print(f"Re-allocated file written to:\n  {out_path}")



In [ ]:
df = pd.read_csv("./multiple_regression/super_linreg_df_2020.csv", dtype={"geoid":str})
df.shape

In [ ]:
df.columns

In [ ]:
# Only keep census variables that also appear (not all nan) in st. Louis' census data
features_in_stl_df = pd.read_csv('./stl/features_in_stl.csv')
features_in_stl = features_in_stl_df['features'].tolist()

dont_drop = ['city', 'trips_per_day', 'total_trips']
keep = []
for feature in df.columns:
    if feature in features_in_stl or feature in dont_drop:
        keep.append(feature)
df_selected = df[keep]
df_selected.shape

In [ ]:
# CLEAN DATA
print("Original shape: " , df.shape)
data = df_selected.dropna(how='all') #thresh = require that many non na values to not be dropped
print("After removing rows missing every entry: " , data.shape)
data = data.dropna(how='all', axis=1) #thresh = how many non na values to not be dropped
print("After removing cols missing every entry: " , data.shape)

exclude = ['geoid', 'city']
# all other columns: attempt to convert to numeric
for col in data.columns:
    if col not in exclude:
        data[col] = pd.to_numeric(data[col], errors='coerce')
print(data.shape)
data = data.dropna(axis=1, how='all')
print(data.shape)

for city, group in data.groupby('city'):
    for col in data.columns:
        if col == 'geoid' or col == 'city':
            continue
        median = group[col].median()
        data.loc[group.index, col] = group[col].fillna(median)
print(data.shape)
data = data.dropna(axis=1, how='any')
print(data.shape)
total_nans = data.isna().sum().sum()
print(f"Total NaNs in DataFrame: {total_nans}")

In [ ]:
city_tpd = df.groupby('city')['trips_per_day'].sum()

In [ ]:
city_tpd

In [ ]:
import pandas as pd

# 1) Make sure denominators are numeric
data['ALAND'] = pd.to_numeric(data['aland'], errors='coerce')
data['S0101_C01_001E'] = pd.to_numeric(data['S0101_C01_001E'], errors='coerce')

# 2) Build “safe” denominators (zeros → 1)
aland_safe = data['ALAND'].replace(0, 1)
pop_safe  = data['S0101_C01_001E'].replace(0, 1)

# 3) Exclude columns you never want to transform
exclude = {
    'geoid', 
    'aland', 
    'total_trips', 
    'trips_per_day', 
    'S0101_C01_001E'
}

# 4) Identify candidate columns
candidates = [col for col in data.columns if col not in exclude]

# 5) Select only those that are fully numeric
fully_numeric = []
for col in candidates:
    conv = pd.to_numeric(data[col], errors='coerce')
    if not conv.isna().any():        # “fully” convertible
        fully_numeric.append(col)
        data[col] = conv              # overwrite with the numeric version

# 6) Create per-area and per-capita columns
for col in fully_numeric:
    data[f'{col}_per_ALAND']   = data[col] / aland_safe
    data[f'{col}_per_capita']  = data[col] / pop_safe


In [ ]:
data.shape

In [ ]:
total_nans = data.isna().sum().sum()
print(f"Total NaNs in DataFrame: {total_nans}")

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# 0.  Identify predictors vs. non-predictors
# -----------------------------------------------------------
NON_FEATURES = ['geoid', 'aland', 'total_trips', 'trips_per_day', 'city']

num_features = [
    c for c in data.columns.difference(NON_FEATURES)
    if pd.api.types.is_numeric_dtype(data[c])
]

# ----------------------------------------------
# 1.  Z-score every numeric column *within city*
# ----------------------------------------------
cols_to_scale = num_features + ['trips_per_day']

def zscore(series):
    sd = series.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        sd = 1.0                  # avoid division-by-zero
    return (series - series.mean()) / sd

data_z = data.copy()
data_z[cols_to_scale] = (
    data.groupby('city')[cols_to_scale]
        .transform(zscore)        # each col handled as a Series
)

In [ ]:
import os
import geopandas as gpd
import pandas as pd

# 0. Ensure data_z GEOIDs are 11-character strings, zero-padded
data_z['geoid'] = data_z['geoid'].astype(str).str.zfill(11)

# 1. Load all tract shapefiles under './tract_shapefiles'
shp_root = "./tract_shapefiles"
gdf_list = []
for state_folder in os.listdir(shp_root):
    folder_path = os.path.join(shp_root, state_folder)
    if not os.path.isdir(folder_path):
        continue
    # find .shp files in each state folder
    for fname in os.listdir(folder_path):
        if fname.endswith(".shp"):
            shp_path = os.path.join(folder_path, fname)
            tmp = gpd.read_file(shp_path)[['GEOID', 'INTPTLON', 'INTPTLAT']].copy()
            # Zero-pad shapefile GEOIDs to 11 characters
            tmp['GEOID'] = tmp['GEOID'].astype(str).str.zfill(11).str[-11:]
            gdf_list.append(tmp)

# 2. Concatenate and prepare the GeoDataFrame
gdf_all = pd.concat(gdf_list, ignore_index=True)
gdf_all = gdf_all.rename(columns={
    'GEOID': 'geoid',
    'INTPTLON': 'centroid_x',
    'INTPTLAT': 'centroid_y'
})
# Ensure numeric types
gdf_all['centroid_x'] = pd.to_numeric(gdf_all['centroid_x'], errors='coerce')
gdf_all['centroid_y'] = pd.to_numeric(gdf_all['centroid_y'], errors='coerce')
# Drop any duplicates by geoid
gdf_all = gdf_all.drop_duplicates(subset='geoid')

# 3. Merge centroids into your data_z DataFrame
data_z = data_z.merge(
    gdf_all[['geoid', 'centroid_x', 'centroid_y']],
    on='geoid',
    how='left'
)


In [ ]:
data_z.shape

In [ ]:
from sklearn.neighbors import NearestNeighbors
# Compute Spatial lags for every column

# 1) Identify which columns to lag (e.g. all numeric, minus metadata)
meta_cols  = ['geoid','city','centroid_x','centroid_y','trips_per_day', 'total_trips', 'aland']
feat_cols  = [c for c in data_z.columns if c not in meta_cols]

# 2) Compute 5‑NN on the centroids
coords     = data_z[['centroid_x','centroid_y']].values
nbrs_model = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs    = nbrs_model.kneighbors(coords)

# 3) Build the lagged‐feature DataFrame
lag_dict = {}
for index, f in enumerate(feat_cols):
    arr = data_z[f].values
    # drop the self‑index nbrs[0], average neighbors 1:6
    lag_dict[f"{f}_lag"] = [arr[nbrs[1:]].mean() for nbrs in idxs]

lags_df = pd.DataFrame(lag_dict, index=data_z.index)

# 4) Concatenate back onto data_z
data_z = pd.concat([data_z, lags_df], axis=1)

# Now data_z has both `f` and `f_lag` for every feature `f` and you can:
#   corrs = ...                      # run your per-city corrwith on data_z
#   selected = corr_filter(corrs, ...) 
#   etc.

In [ ]:
data_z.shape

In [ ]:
NON_FEATURES = ['geoid', 'aland', 'total_trips', 'trips_per_day', 'city']

num_features = [
    c for c in data_z.columns.difference(NON_FEATURES)
    if pd.api.types.is_numeric_dtype(data_z[c])
]

In [ ]:
len(num_features)

In [ ]:
# # ------------------------------------------------------------
# # Incorporating Pearson or Spearman with tunable thresholds
# # ------------------------------------------------------------
# import numpy as np
# import pandas as pd

# # ----- PARAMETERS -----
# CORR_METHOD = 'spearman'       # 'pearson' or 'spearman'

# # Thresholds for Pearson-based filtering
# KEEP_THRESH_PEARSON = 0.20
# WINS_THRESH_PEARSON = 10

# # Thresholds for Spearman-based filtering
# KEEP_THRESH_SPEARMAN = 0.25
# WINS_THRESH_SPEARMAN = 9

# # Redundancy threshold
# REDUNDANCY_R = 0.9

# # ----------------------  
# # 1) Define the filtering function
# # ----------------------
# def corr_filter(corrs: pd.DataFrame, keep_thresh: float, wins_thresh: int):
#     """
#     Returns features with
#       – mean(|r|) ≥ keep_thresh
#       – AND wins (cities with |r| ≥ keep_thresh) ≥ wins_thresh
#     """
#     abs_corrs = corrs.abs()
#     summary = abs_corrs.agg(['mean','median'], axis=1).rename(
#         columns={'mean':'mean_abs_r', 'median':'median_abs_r'}
#     )
#     summary['wins'] = (abs_corrs >= keep_thresh).sum(axis=1)
#     # Select features meeting both criteria
#     return summary.query("mean_abs_r >= @keep_thresh and wins >= @wins_thresh").index.tolist()

# # ----------------------
# # 2) Compute per-city correlations
# # ----------------------
# # data_z: your z-scored DataFrame
# # num_features: list of feature column names
# # 'trips_per_day' is the target column

# # Build a dict of city -> correlation Series
# corr_dict = {}
# for city, g in data_z.groupby('city', sort=False):
#     if CORR_METHOD.lower() == 'pearson':
#         corr_vec = g[num_features].corrwith(g['trips_per_day']).dropna()
#     elif CORR_METHOD.lower() == 'spearman':
#         # Compute Spearman by ranking before Pearson
#         corr_vec = g[num_features].apply(
#             lambda col: col.corr(g['trips_per_day'], method='spearman')
#         ).dropna()
#     else:
#         raise ValueError("CORR_METHOD must be 'pearson' or 'spearman'")
#     corr_dict[city] = corr_vec

# # Assemble into a DataFrame: rows=features, cols=cities
# corrs = pd.DataFrame(corr_dict).astype(float)

# # ----------------------
# # 3) Apply the filter with appropriate thresholds
# # ----------------------
# if CORR_METHOD.lower() == 'pearson':
#     keep_thresh = KEEP_THRESH_PEARSON
#     wins_thresh = WINS_THRESH_PEARSON
# else:
#     keep_thresh = KEEP_THRESH_SPEARMAN
#     wins_thresh = WINS_THRESH_SPEARMAN

# # 3a) Correlation + wins filter
# selected = corr_filter(corrs, keep_thresh, wins_thresh)
# print(f"After {CORR_METHOD} corr+ wins: {len(selected)} features selected")

# # 3b) Drop collinear features
# upper = (
#     data_z[selected]
#       .corr().abs()
#       .where(np.triu(np.ones((len(selected), len(selected)), dtype=bool), k=1))
# )
# to_drop = [c for c in upper.columns if any(upper[c] > REDUNDANCY_R)]
# selected = [c for c in selected if c not in to_drop]
# print(f"After dropping collinear: {len(selected)} features remain")

# # 3c) Inspect top features by mean_abs_r
# abs_corrs = corrs.abs()
# summary = abs_corrs.agg(['mean','median'], axis=1).rename(
#     columns={'mean':'mean_abs_r','median':'median_abs_r'}
# )
# summary['wins'] = (abs_corrs >= keep_thresh).sum(axis=1)

# top10 = summary.loc[selected].sort_values('mean_abs_r', ascending=False).head(10)
# print("\nTop 10 features by mean_abs_r:")
# for feat, row in top10.iterrows():
#     print(f"{feat:30s} mean_abs_r = {row.mean_abs_r:.3f} wins = {int(row.wins)}")



In [ ]:
import numpy as np
import pandas as pd

def filter_features(
    corrs: pd.DataFrame,
    data_z: pd.DataFrame,
    keep_thresh: float,
    wins_thresh: int,
    colinear_thresh: float,
    same_sign_frac: float = 1.0
) -> list:

    # Runs four filters in sequence, returning the list of features that survive:
    #   1) mean |r| ≥ keep_thresh AND count(|r| ≥ keep_thresh) ≥ wins_thresh
    #   2) drop any pair with |corr| > colinear_thresh (keep only one of each pair)
    #   3) keep only features whose sign of r is the same in ≥ same_sign_frac fraction of cities

    # 1) corr + wins filter
    abs_corrs = corrs.abs()
    summary = (
        abs_corrs
          .agg(['mean','median'], axis=1)
          .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
          .assign(wins=(abs_corrs >= keep_thresh).sum(axis=1))
    )
    sel = summary.query(
        "mean_abs_r >= @keep_thresh and wins >= @wins_thresh"
    ).index.tolist()
    print(f'{len(sel)} features after filtering. KEEP_THRESH = {keep_thresh}. WINS_THRESH = {wins_thresh}')
    if len(sel) < 1:
        return []

    # 2) drop collinear pairs
    upper = (
        data_z[sel]
          .corr().abs()
          .where(np.triu(np.ones((len(sel), len(sel)), dtype=bool), k=1))
    )
    to_drop = [c for c in upper.columns if any(upper[c] > colinear_thresh)]
    sel = [c for c in sel if c not in to_drop]
    
    if len(sel) < 1:
        return []
    print(f'{len(sel)} features after collinear filter')
    # 3) same-sign filter
    corrs_sel = corrs.loc[sel]
    signs_sel = corrs_sel.apply(np.sign)
    modal_sign = signs_sel.mode(axis=1)[0]
    sign_frac = signs_sel.eq(modal_sign, axis=0).sum(axis=1) / signs_sel.shape[1]
    sel = sign_frac[sign_frac >= same_sign_frac].index.tolist()
    print(f'{len(sel)} features after same-sign filter. Fraction = {same_sign_frac}')

    return sel


In [ ]:
corr_dict = {
    city: g[num_features].corrwith(g['trips_per_day']).dropna()
    for city, g in data_z.groupby('city', sort=False)
}
corrs = pd.DataFrame(corr_dict).astype(float)

In [ ]:
data_z.shape

In [ ]:
corrs.to_csv('./corr_outputs/all_correlations.csv')

In [ ]:

# KEEP_THRESH = 0.20
# WINS_THRESH = 10

# irrelevant
#{'keep': 0.1, 'wins': 5, 'alpha': 0.01, 'l1_ratio': 0.5, 'r2': 0.3179791943279593, 'corr_feats': 276, 'collinear_removed': 139, 'same_sign': 38, 'lasso_feats': 22}


# ElasticNet Best Thresholds:
#{'keep': 0.1, 'wins': 7, 'corr_feats': 1530, 'collinear_removed': 593, 'same_sign': 112, 'lasso_feats': 33, 'r2': 0.3501862574366603}
# {'keep': 0.1, 'wins': 7, 'corr_feats': 1530, 'collinear_removed': 593, 'same_sign': 112, 'lasso_feats': 33, 'r2': 0.3501862574366603}
# {'keep': 0.25, 'wins': 7, 'alpha': 0.01, 'l1_ratio': 0.8, 'r2': 0.38222665798173155, 'corr_feats': 661, 'collinear_removed': 314, 'same_sign': 209, 'lasso_feats': 55}
# {'keep': 0.1, 'wins': 9, 'alpha': 0.01, 'l1_ratio': 0.5, 'r2': 0.33728442700537814, 'corr_feats': 62, 'lasso_feats': 23}
# with log {'keep': 0.1, 'wins': 5, 'alpha': 0.001, 'l1_ratio': 1.0, 'r2': 0.4140527445186853, 'corr_feats': 121, 'lasso_feats': 53}

KEEP_THRESH = 0.1
WINS_THRESH = 6

# Random Forest Best Thresholds:
#{'keep_thresh': 0.1, 'wins_thresh': 10, 'n_corr_feats': 365, 'n_collinear_removed': 168, 'n_same_sign': 83, 'n_lasso_feats': 38, 'r2_with_lags': 0.3511735351772509}
#{'keep_thresh': 0.25, 'wins_thresh': 8, 'n_corr_feats': 48, 'n_collinear_removed': 25, 'n_same_sign': 17, 'n_lasso_feats': 12, 'r2_with_lags': 0.32066904191935475}

# KEEP_THRESH = 0.25
# WINS_THRESH = 8

# # HGB Best Thresholds 
# {'keep_thresh': 0.15, 'wins_thresh': 7, 'n_corr_feats': 523, 'n_collinear_removed': 239, 'n_lasso_feats': 34, 'r2_with_lags': 0.27677731135638256}
# {'keep_thresh': 0.15, 'wins_thresh': 7, 'n_corr_feats': 523, 'n_collinear_removed': 239, 'n_lasso_feats': 34, 'r2_with_lags': 0.27677731135638256}
#{'hgb_keep': 0.1, 'hgb_wins': 5, 'learning_rate': 0.1, 'max_depth': 5, 'max_iter': 100, 'r2': 0.3782833030650996, 'corr_feats': 121, 'lasso_feats': 53}

# KEEP_THRESH = 0.1
# WINS_THRESH = 5


REDUNDANCY_R    = 0.90
SAME_SIGN_FRAC  = 1.0


selected = filter_features(
    corrs, data_z,
    keep_thresh=KEEP_THRESH,
    wins_thresh=WINS_THRESH,
    colinear_thresh=REDUNDANCY_R,
    same_sign_frac=SAME_SIGN_FRAC
)
print(len(selected), "Features surviving all filters:", selected)


In [ ]:
# Setup for cv stuff
# ============================================================
#  A.  Imports & global cache (picklable)
# ============================================================
import numpy as np, pandas as pd, functools, joblib, time
from sklearn.base                import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline            import Pipeline
from sklearn.preprocessing       import StandardScaler
from sklearn.model_selection     import GroupKFold, RandomizedSearchCV, ParameterSampler, GroupKFold, cross_val_score
from sklearn.linear_model        import ElasticNet, ElasticNetCV, Ridge
from sklearn.ensemble            import HistGradientBoostingRegressor, StackingRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics             import r2_score
from scipy.stats                 import uniform, loguniform     # ⇐ for random grids

# …  your own helpers …
# →  data_z, corrs, filter_features, REDUNDANCY_R, SAME_SIGN_FRAC, NUMERIC_FEATURES

# ----------------  picklable correlation‑filter cache  -----------------
_FILTER_CACHE = {}
def corr_filter_cached(keep, wins, verbose=False):
    key = (int(round(keep*100)), int(wins))
    if key not in _FILTER_CACHE:
        cols = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
        _FILTER_CACHE[key] = cols
        if verbose:
            print(f"{len(cols)} cols after filter (keep={keep}, wins={wins})")
    return _FILTER_CACHE[key]

# ============================================================
#  B.  Small transformers
# ============================================================
class CorrFilter(BaseEstimator, TransformerMixin):
    def __init__(self, keep=.15, wins=6, verbose=False):
        self.keep, self.wins, self.verbose = keep, wins, verbose
    def fit(self, X, y=None):
        self.cols_ = corr_filter_cached(self.keep, self.wins, self.verbose)
        return self
    def transform(self, X):   return X.loc[:, self.cols_]

class ENetSelector(BaseEstimator, TransformerMixin):
    def __init__(self, sel_alphas=(0.001,0.01,0.1), sel_l1=(0.2,0.5,0.8)):
        self.sel_alphas, self.sel_l1 = sel_alphas, sel_l1
    def fit(self, X, y):
        self.enet_ = ElasticNetCV(
            alphas      = self.sel_alphas,
            l1_ratio    = self.sel_l1,
            cv          = 5,
            random_state= 42,
            n_jobs      = 1                  # avoid nested threading
        ).fit(X, y)
        self.cols_ = X.columns[self.enet_.coef_ != 0].tolist()
        return self
    def transform(self, X):   return X.loc[:, self.cols_]

# ============================================================
#  C.  Base‑learner factory
# ============================================================
def make_base(model_template):
    return Pipeline([
        ('filter', CorrFilter()),
        ('select', ENetSelector()),
        ('scale',  StandardScaler()),
        ('model',  clone(model_template))
    ])

enet_tpl  = Pipeline([('enet',  ElasticNet(max_iter=10_000, random_state=42))])
hgb_tpl   = Pipeline([('hgb',   HistGradientBoostingRegressor(random_state=42))])
ridge_tpl = Pipeline([('ridge', Ridge())])
pls_tpl   = Pipeline([('pls',   PLSRegression())])

stack = StackingRegressor(
    estimators=[
        ('enet',  make_base(enet_tpl)),
        ('hgb',   make_base(hgb_tpl)),
        ('ridge', make_base(ridge_tpl)),
        ('pls',   make_base(pls_tpl)),
    ],
    final_estimator = ElasticNet(max_iter=10_000, random_state=42),
    n_jobs = -1
)

full_pipe = Pipeline([('stk', stack)])

# ============================================================
#  D.  Random‑search space   (tune the ranges as you like)
# ============================================================
param_dists = {
    # corr keep / wins  (uniform discrete draws)
    'stk__enet__filter__keep'  : uniform(0.10, 0.20),   # 0.10‑0.30
    'stk__enet__filter__wins'  : [6, 8, 10],
    'stk__hgb__filter__keep'   : uniform(0.10, 0.20),
    'stk__hgb__filter__wins'   : [6, 8, 10],
    'stk__ridge__filter__keep' : uniform(0.10, 0.20),
    'stk__ridge__filter__wins' : [6, 8, 10],
    'stk__pls__filter__keep'   : uniform(0.10, 0.20),
    'stk__pls__filter__wins'   : [6, 8, 10],

    # selector ENet grids
    'stk__enet__select__sel_alphas' : [[1e-3, 1e-2, 1e-1]],
    'stk__enet__select__sel_l1'     : [[0.2, 0.5, 0.8]],

    # final model hyper‑params
    'stk__enet__model__enet__alpha'    : loguniform(1e-3, 1e0),
    'stk__enet__model__enet__l1_ratio' : uniform(0.2, 0.6),
    'stk__hgb__model__hgb__learning_rate': uniform(0.03, 0.17),   # 0.03‑0.20
    'stk__hgb__model__hgb__max_depth'    : [3, 5, 7],
    'stk__ridge__model__ridge__alpha'    : loguniform(1e-3, 1e1),
    'stk__pls__model__pls__n_components' : [5, 10, 15],

    # meta learner
    'stk__final_estimator__alpha'       : loguniform(1e-3, 1e0),
    'stk__final_estimator__l1_ratio'    : uniform(0.2, 0.6),
}

# slimmed_param_dists — *one or two* key params per learner
slimmed_param_dists = {
    'stk__enet__filter__keep'  : [0.15, 0.25],
    'stk__enet__filter__wins'  : [6, 9],
    'stk__enet__model__enet__alpha'    : loguniform(1e-2, 3e-1),
    'stk__enet__model__enet__l1_ratio' : uniform(0.3, 0.5),   # 0.3–0.8

    'stk__hgb__model__hgb__learning_rate': uniform(0.05, 0.15),
    'stk__hgb__model__hgb__max_depth'    : [3, 5],

    'stk__ridge__model__ridge__alpha'    : loguniform(1e-2, 1e1),
    'stk__pls__model__pls__n_components' : [5, 10],

    'stk__final_estimator__alpha'        : loguniform(1e-2, 3e-1),
    'stk__final_estimator__l1_ratio'     : uniform(0.3, 0.5),
}


# how many random samples?
N_RANDOM = 100

# ============================================================
#  E.  Data & CV objects
# ============================================================
X       = data_z[num_features]          # already incl. lags
y       = data_z['trips_per_day'].values
groups  = data_z['city'].values
cities  = np.unique(groups)

outer_cv = GroupKFold(n_splits=len(cities))

inner_cv = GroupKFold(n_splits=5)

# ============================================================
#  F.  Outer LOCO loop with RandomizedSearchCV inside
# ============================================================
outer_scores         = []
base_scores_by_city  = {m: [] for m in ['enet','hgb','ridge','pls']}

fold = 0
for train_idx, test_idx in outer_cv.split(X, groups=groups):
    fold += 1
    holdout_city = data_z.iloc[test_idx]['city'].iloc[0]
    print(f"\n==========  OUTER fold {fold}/{len(cities)} | hold‑out = {holdout_city}  ==========")

    rnd = RandomizedSearchCV(
        estimator      = full_pipe,
        param_distributions = slimmed_param_dists,
        n_iter         = N_RANDOM,
        cv             = inner_cv.split(X.iloc[train_idx], groups=groups[train_idx]),
        scoring        = 'r2',
        n_jobs         = -1,
        verbose        = 2,
        random_state   = 42
    )

    with joblib.parallel_backend('loky', inner_max_num_threads=1):
        rnd.fit(X.iloc[train_idx], y[train_idx])

    best_model = rnd.best_estimator_
    y_pred     = best_model.predict(X.iloc[test_idx])
    r2_outer   = r2_score(y[test_idx], y_pred)
    outer_scores.append(r2_outer)
    print(f"→ Ensemble R² on {holdout_city}: {r2_outer:.3f}")

    # score base learners on hold‑out
    for name, base in best_model.named_steps['stk'].named_estimators_.items():
        r2_base = r2_score(y[test_idx], base.predict(X.iloc[test_idx]))
        base_scores_by_city[name].append((holdout_city, r2_base))
        print(f"   {name:5s} R² = {r2_base:.3f}")

# ============================================================
#  G.  Summary
# ============================================================
outer_scores = np.array(outer_scores)
print("\n================  SUMMARY  =================")
print(f"Nested LOCO R² (ensemble) = {outer_scores.mean():.4f} ± {outer_scores.std(ddof=1):.4f}")

for name, vals in base_scores_by_city.items():
    df = pd.DataFrame(vals, columns=['city','r2']).set_index('city')
    print(f"\n{name.upper()}  mean R² = {df.r2.mean():.3f}")
    print(df.r2.to_string())

# ============================================================
#  H.  Refit on all data (optional)
# ============================================================
rnd_final = RandomizedSearchCV(
    estimator = full_pipe,
    param_distributions = slimmed_param_dists,
    n_iter    = N_RANDOM,
    cv        = inner_cv,
    scoring   = 'r2',
    n_jobs    = -1,
    verbose   = 2,
    random_state = 42
)
with joblib.parallel_backend('loky', inner_max_num_threads=1):
    rnd_final.fit(X, y, groups=groups)

final_stack = rnd_final.best_estimator_
print("\nBest params (all data):")
print(rnd_final.best_params_)

city_r2 = {
    city: r2_score(y[data_z['city'] == city],
                   final_stack.predict(X[data_z['city'] == city]))
    for city in cities
}
print("\nFinal model R² by city:")
print(pd.Series(city_r2).to_string())


In [ ]:
X = data_z[num_features]

In [ ]:
from sklearn.model_selection import ParameterSampler, GroupKFold, cross_val_score
from sklearn.base            import clone
from sklearn.metrics         import r2_score
import numpy as np
import pandas as pd

# --------------------------------------------------
# A) prepare data & CV objects
# --------------------------------------------------
X        = data_z[num_features]          
y        = data_z['trips_per_day'].values
groups   = data_z['city'].values
cities   = np.unique(groups)

outer_cv = GroupKFold(n_splits=len(cities))
inner_cv = GroupKFold(n_splits=5) 

# how many inner candidates?
param_list = list(ParameterSampler(
    slimmed_param_dists,
    n_iter       = N_RANDOM,
    random_state = 42
))
N_INNER = len(param_list)

outer_scores        = []
base_scores_by_city = {m: [] for m in ['enet','hgb','ridge','pls']}

# NEW: dict to hold each city's best hyperparameters
best_params_by_city = {}

# --------------------------------------------------
# B) nested LOCO with manual random search
# --------------------------------------------------
for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, groups=groups), start=1):

    holdout_city = data_z.iloc[test_idx]['city'].iloc[0]
    print(f"\n=== OUTER fold {fold_idx}/{len(cities)} : hold‐out = {holdout_city} ===")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx],    y[test_idx]
    grp_train       = groups[train_idx]

    # inner random search
    best_inner_r2 = -np.inf
    best_params   = None

    for inner_idx, params in enumerate(param_list, start=1):
        print(f"  → INNER trial {inner_idx}/{N_INNER}")
        print("      params:", {
            k:v for k,v in params.items()
            if 'filter' in k or 'model' in k and any(n in k for n in ['alpha','l1','depth','n_components'])
        })

        pipe = clone(full_pipe).set_params(**params)
        scores = cross_val_score(
            pipe,
            X_train, y_train,
            cv      = inner_cv.split(X_train, groups=grp_train),
            scoring = 'r2',
            n_jobs  = -1
        )
        mean_r2 = scores.mean()
        print(f"      → mean inner R² = {mean_r2:.4f}")

        if mean_r2 > best_inner_r2:
            best_inner_r2, best_params = mean_r2, params

    # end inner trials
    print(f"  *** Best inner R² = {best_inner_r2:.4f} with params:")
    print(best_params)

    # SAVE the best_params for this city
    best_params_by_city[holdout_city] = best_params

    # re‐fit on full train with best_params
    final_model = clone(full_pipe).set_params(**best_params)
    final_model.fit(X_train, y_train)

    # evaluate on the hold‐out city
    y_pred    = final_model.predict(X_test)
    outer_r2  = r2_score(y_test, y_pred)
    outer_scores.append(outer_r2)
    print(f"→ Outer R² on {holdout_city} = {outer_r2:.4f}")

    # also score each base learner
    for name, base in final_model.named_steps['stk'].named_estimators_.items():
        r2_base = r2_score(y_test, base.predict(X_test))
        base_scores_by_city[name].append((holdout_city, r2_base))
        print(f"   {name:5s} R² = {r2_base:.4f}")

# --------------------------------------------------
# C) summary
# --------------------------------------------------
outer_scores = np.array(outer_scores)
print(f"\n=== Nested LOCO R² (ensemble) = {outer_scores.mean():.4f} ± {outer_scores.std(ddof=1):.4f} ===")

for name, vals in base_scores_by_city.items():
    df = pd.DataFrame(vals, columns=['city','r2']).set_index('city')
    print(f"\n{name.upper()} mean R² = {df.r2.mean():.4f}")
    print(df.r2.to_string())

# Now you can inspect all per‐city best_params:
print("\nBest hyperparameters by city:")
for city, params in best_params_by_city.items():
    print(f" - {city}: {params}")


In [ ]:
"""
Nested LOCO-CV   ·   ENet + HGB stacking ensemble
-------------------------------------------------
Assumes the workspace already defines
  • data_z              : DataFrame with features + 'trips_per_day' + 'city'
  • corrs               : feature × city correlation DataFrame
  • filter_features(...) helper
  • REDUNDANCY_R, SAME_SIGN_FRAC, NUMERIC_FEATURES
"""

# ============================================================
#  A.  Imports
# ============================================================
import numpy as np, pandas as pd, joblib
from sklearn.base                import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline            import Pipeline
from sklearn.preprocessing       import StandardScaler
from sklearn.model_selection     import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.linear_model        import ElasticNet, ElasticNetCV
from sklearn.ensemble            import HistGradientBoostingRegressor, StackingRegressor
from sklearn.metrics             import r2_score
from scipy.stats                 import uniform, loguniform

# ============================================================
#  B.  Correlation filter + selector
# ============================================================
_FILTER_CACHE = {}
def corr_filter_cached(keep, wins):
    key = (int(round(keep*100)), int(wins))
    if key not in _FILTER_CACHE:
        _FILTER_CACHE[key] = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
    return _FILTER_CACHE[key]

class CorrFilter(BaseEstimator, TransformerMixin):
    def __init__(self, keep=.15, wins=6):   self.keep, self.wins = keep, wins
    def fit(self, X, y=None):
        self.cols_ = corr_filter_cached(self.keep, self.wins)
        return self
    def transform(self, X):                 return X.loc[:, self.cols_]

class ENetSelector(BaseEstimator, TransformerMixin):
    def __init__(self, alphas=(1e-3,1e-2,1e-1), l1=(.2,.5,.8)):
        self.alphas, self.l1 = alphas, l1
    def fit(self, X, y):
        self.est_ = ElasticNetCV(
            alphas=self.alphas, l1_ratio=self.l1,
            cv=5, random_state=42, n_jobs=1
        ).fit(X, y)
        self.cols_ = X.columns[self.est_.coef_ != 0].tolist()
        return self
    def transform(self, X):                 return X.loc[:, self.cols_]

# ============================================================
#  C.  Pipeline factory
# ============================================================
def make_base(model_tpl):
    return Pipeline([
        ('filter', CorrFilter()),
        ('select', ENetSelector()),
        ('scale',  StandardScaler()),
        ('model',  clone(model_tpl))
    ])

enet_tpl = Pipeline([('enet', ElasticNet(max_iter=10_000, random_state=42))])
hgb_tpl  = Pipeline([('hgb',  HistGradientBoostingRegressor(random_state=42))])

stack = StackingRegressor(
    estimators=[
        ('enet', make_base(enet_tpl)),
        ('hgb',  make_base(hgb_tpl)),
    ],
    final_estimator = ElasticNet(max_iter=10_000, random_state=42),
    n_jobs = -1
)
full_pipe = Pipeline([('stk', stack)])

# ============================================================
#  D.  Hyper-parameter space  (trimmed)
# ============================================================
param_dists = {
    # corr filter
    'stk__enet__filter__keep' : uniform(0.10, 0.20),
    'stk__enet__filter__wins' : [6, 8, 10],
    'stk__hgb__filter__keep'  : uniform(0.10, 0.20),
    'stk__hgb__filter__wins'  : [6, 8, 10],

    # ENet base learner
    'stk__enet__model__enet__alpha'    : loguniform(1e-3, 1e0),
    'stk__enet__model__enet__l1_ratio' : uniform(0.2, 0.6),

    # HGB base learner
    'stk__hgb__model__hgb__learning_rate' : uniform(0.03, 0.17),
    'stk__hgb__model__hgb__max_depth'     : [3, 5, 7],

    # meta-learner
    'stk__final_estimator__alpha'     : loguniform(1e-3, 1e0),
    'stk__final_estimator__l1_ratio'  : uniform(0.2, 0.6),
}

N_RANDOM = 100

# ============================================================
#  E.  Data & CV setup
# ============================================================
num_features = NUMERIC_FEATURES               # already includes spatial lags etc.
X       = data_z[num_features]
y       = data_z['trips_per_day'].values
groups  = data_z['city'].values
cities  = np.unique(groups)

outer_cv = GroupKFold(n_splits=len(cities))
inner_cv = GroupKFold(n_splits=5)

# ============================================================
#  F.  Nested LOCO-CV
# ============================================================
outer_scores = []
base_scores  = {m: [] for m in ['enet','hgb']}

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, groups=groups), start=1):
    holdout = data_z.iloc[test_idx]['city'].iloc[0]
    print(f"\n=== OUTER fold {fold}/{len(cities)}  |  hold-out city = {holdout} ===")

    search = RandomizedSearchCV(
        estimator          = full_pipe,
        param_distributions= param_dists,
        n_iter             = N_RANDOM,
        cv                 = inner_cv.split(X.iloc[train_idx], groups=groups[train_idx]),
        scoring            = 'r2',
        n_jobs             = -1,
        random_state       = 42,
        verbose            = 2
    )

    with joblib.parallel_backend('loky', inner_max_num_threads=1):
        search.fit(X.iloc[train_idx], y[train_idx])

    best_model = search.best_estimator_
    y_pred     = best_model.predict(X.iloc[test_idx])
    r2_out     = r2_score(y[test_idx], y_pred)
    outer_scores.append(r2_out)
    print(f"→ Ensemble R² on {holdout}: {r2_out:.4f}")

    # base learners
    for name, base in best_model.named_steps['stk'].named_estimators_.items():
        r2_b = r2_score(y[test_idx], base.predict(X.iloc[test_idx]))
        base_scores[name].append((holdout, r2_b))
        print(f"   {name.upper():4s} R² = {r2_b:.4f}")

# ============================================================
#  G.  Summary
# ============================================================
outer_scores = np.array(outer_scores)
print("\n===== SUMMARY =====")
print(f"Nested LOCO R² (ensemble) = {outer_scores.mean():.4f} ± {outer_scores.std(ddof=1):.4f}")

for name, vals in base_scores.items():
    df = pd.DataFrame(vals, columns=['city','r2']).set_index('city')
    print(f"\n{name.upper()} mean R² = {df.r2.mean():.4f}")
    print(df.r2.to_string())


In [ ]:
# --------------------------------------------
# 1) Imports & memory cache
# --------------------------------------------
import numpy as np
import pandas as pd

from joblib import Memory, Parallel, delayed, parallel_backend
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV, GroupKFold
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import r2_score
from sklearn.linear_model    import ElasticNet
from sklearn.ensemble        import HistGradientBoostingRegressor, StackingRegressor

# assume your make_base, enet_tpl, hgb_tpl, slimmed_param_dists, data_z, num_features defined as before

# set up disk cache for fitted transformers
memory = Memory(location='cache_dir', verbose=0)


# --------------------------------------------
# 2) Build a reduced stacking pipeline
# --------------------------------------------
stack2 = StackingRegressor(
    estimators=[
        ('enet', make_base(enet_tpl)),
        ('hgb',  make_base(hgb_tpl)),
    ],
    final_estimator = ElasticNet(max_iter=10_000, random_state=42),
    n_jobs = 1   # keep inner parallelism controlled by HalvingRandomSearchCV
)
full_pipe = Pipeline([('stk', stack2)], memory=memory)


# --------------------------------------------
# 3) Filter param grid to ENet + HGB only
# --------------------------------------------
param_dists_2 = {
    k: v for k, v in slimmed_param_dists.items()
    if k.startswith('stk__enet__') or k.startswith('stk__hgb__')
}


# --------------------------------------------
# 4) CV definitions
# --------------------------------------------
X      = data_z[num_features]
y      = data_z['trips_per_day'].values
groups = data_z['city'].values
cities = np.unique(groups)

outer_cv = GroupKFold(n_splits=len(cities))
inner_cv = GroupKFold(n_splits=5)


def run_fold(train_idx, test_idx):
    city      = data_z.iloc[test_idx]['city'].iloc[0]
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y[train_idx],    y[test_idx]
    groups_tr  = groups[train_idx]

    halving = HalvingRandomSearchCV(
        estimator           = full_pipe,
        param_distributions = param_dists_2,
        n_candidates        = 50,           # NOTE: renamed from max_candidates
        factor              = 3,
        resource            = 'n_samples',
        cv                  = inner_cv,
        scoring             = 'r2',
        n_jobs              = -1,
        random_state        = 42,
        verbose             = 2
    )

    with parallel_backend('loky', inner_max_num_threads=1):
        halving.fit(X_tr, y_tr, groups=groups_tr)

    best    = halving.best_estimator_
    y_pred  = best.predict(X_te)
    outer_r2 = r2_score(y_te, y_pred)

    base_r2 = {
        name: r2_score(y_te, mdl.predict(X_te))
        for name, mdl in best.named_steps['stk'].named_estimators_.items()
    }

    return city, outer_r2, base_r2, halving.best_params_


# --------------------------------------------
# 6) Parallel outer‐fold execution
# --------------------------------------------
results = Parallel(n_jobs=-1)(
    delayed(run_fold)(tr, te)
    for tr, te in outer_cv.split(X, groups=groups)
)


# --------------------------------------------
# 7) Collate & report
# --------------------------------------------
best_params_by_city = {}
outer_scores = []
base_scores_by_city = {'enet': [], 'hgb': []}

for city, r2_val, base_dict, bp in results:
    print(f"{city:10s} → ensemble R² = {r2_val:.3f}")
    outer_scores.append(r2_val)
    best_params_by_city[city] = bp

    for name, sc in base_dict.items():
        print(f"   {name:5s} R² = {sc:.3f}")
        base_scores_by_city[name].append((city, sc))

print(f"\nOverall nested‐LOCO R² = {np.mean(outer_scores):.3f} ± {np.std(outer_scores, ddof=1):.3f}")

In [ ]:
# → Outer R² on austin = 0.3750
# → Outer R² on boston = 0.4678
# → Outer R² on chicago = 0.5107
# → Outer R² on columbus = 0.2523
# → Outer R² on dmv = 0.5284
# → Outer R² on la = 0.2679
# → Outer R² on nyc = 0.2381
# → Outer R² on philly = 0.4654
# → Outer R² on pittsburgh = 0.2566
# → Outer R² on portland = 0.2826
# → Outer R² on sf = 0.3210
# → Outer R² on austin = 0.3750
#    enet  R² = 0.2797
#    hgb   R² = 0.3589
#    ridge R² = 0.2976
#    pls   R² = 0.2949

# === Nested LOCO R² (ensemble) = 0.3605 ± 0.1128 ===

# ENET mean R² = 0.2682
# city
# nyc          -0.432408
# chicago       0.378649
# dmv           0.554255
# boston        0.473540
# sf            0.265123
# portland      0.328803
# la            0.274438
# philly        0.484061
# columbus      0.085298
# pittsburgh    0.258913
# austin        0.279737

# HGB mean R² = 0.3577
# city
# nyc           0.334175
# chicago       0.543308
# dmv           0.540580
# boston        0.448484
# sf            0.367925
# portland      0.248953
# la            0.226376
# philly        0.394152
# columbus      0.335003
# pittsburgh    0.137015
# austin        0.358884

# RIDGE mean R² = 0.2902
# city
# nyc          -0.327433
# chicago       0.490029
# dmv           0.557993
# boston        0.452325
# sf            0.248435
# portland      0.210193
# la            0.272064
# philly        0.512291
# columbus      0.181459
# pittsburgh    0.297629
# austin        0.297642

# PLS mean R² = 0.2942
# city
# nyc          -0.293815
# chicago       0.491200
# dmv           0.568191
# boston        0.451919
# sf            0.247270
# portland      0.211368
# la            0.276358
# philly        0.509207
# columbus      0.175485
# pittsburgh    0.303603
# austin        0.294873

In [ ]:
(0.321 + 0.2826 + 0.2566 + 0.4654 + 0.2381 + 0.2679 + 0.5284 + 0.2523 + 0.5107 + 0.4678 + 0.3750) / 11

In [ ]:
best_params_by_city = {
    'austin': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.049833191601257244, 'stk__enet__model__enet__l1_ratio': 0.35979712296915084, 'stk__final_estimator__alpha': 0.1131225105716033, 'stk__final_estimator__l1_ratio': 0.6803925243084488, 'stk__hgb__model__hgb__learning_rate': 0.13419157963542444, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.30296104428212484},
    'boston': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.017404995100538886, 'stk__enet__model__enet__l1_ratio': 0.3352843737002149, 'stk__final_estimator__alpha': 0.0889060202841461, 'stk__final_estimator__l1_ratio': 0.3132556552708109, 'stk__hgb__model__hgb__learning_rate': 0.13786633719101948, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.5326262097599814},
    'chicago': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 9, 'stk__enet__model__enet__alpha': 0.2521267904777921, 'stk__enet__model__enet__l1_ratio': 0.7828160165372797, 'stk__final_estimator__alpha': 0.1563510870813346, 'stk__final_estimator__l1_ratio': 0.45230688458668533, 'stk__hgb__model__hgb__learning_rate': 0.06465081710095757, 'stk__hgb__model__hgb__max_depth': 5, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.2091498132903561},
    'columbus': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 9, 'stk__enet__model__enet__alpha': 0.11664190932505923, 'stk__enet__model__enet__l1_ratio': 0.6600182682730371, 'stk__final_estimator__alpha': 0.08852232173650892, 'stk__final_estimator__l1_ratio': 0.64697422223355, 'stk__hgb__model__hgb__learning_rate': 0.13140866650213942, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 5, 'stk__ridge__model__ridge__alpha': 0.10891507139853285},
    'dmv': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.049833191601257244, 'stk__enet__model__enet__l1_ratio': 0.35979712296915084, 'stk__final_estimator__alpha': 0.1131225105716033, 'stk__final_estimator__l1_ratio': 0.6803925243084488, 'stk__hgb__model__hgb__learning_rate': 0.13419157963542444, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.30296104428212484},
    'la': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.017404995100538886, 'stk__enet__model__enet__l1_ratio': 0.3352843737002149, 'stk__final_estimator__alpha': 0.0889060202841461, 'stk__final_estimator__l1_ratio': 0.3132556552708109, 'stk__hgb__model__hgb__learning_rate': 0.13786633719101948, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.5326262097599814},
    'nyc': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 9, 'stk__enet__model__enet__alpha': 0.025793585182259916, 'stk__enet__model__enet__l1_ratio': 0.3885052421383734, 'stk__final_estimator__alpha': 0.013521485225728888, 'stk__final_estimator__l1_ratio': 0.3603179355503004, 'stk__hgb__model__hgb__learning_rate': 0.11911681520490887, 'stk__hgb__model__hgb__max_depth': 5, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.12382535539667001},
    'philly': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.05448281849002502, 'stk__enet__model__enet__l1_ratio': 0.34346014404371183, 'stk__final_estimator__alpha': 0.0621400343470248, 'stk__final_estimator__l1_ratio': 0.5934205590104396, 'stk__hgb__model__hgb__learning_rate': 0.1618159211276495, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.02414020247400934},
    'pittsburgh': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 9, 'stk__enet__model__enet__alpha': 0.26420950301415247, 'stk__enet__model__enet__l1_ratio': 0.7179900602561029, 'stk__final_estimator__alpha': 0.10666901691534954, 'stk__final_estimator__l1_ratio': 0.5044764722071349, 'stk__hgb__model__hgb__learning_rate': 0.07599414801062687, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 5, 'stk__ridge__model__ridge__alpha': 0.05632856590781128},
    'portland': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.21319681991837552, 'stk__enet__model__enet__l1_ratio': 0.4919456106866057, 'stk__final_estimator__alpha': 0.06351751286408563, 'stk__final_estimator__l1_ratio': 0.7532360554822735, 'stk__hgb__model__hgb__learning_rate': 0.1436356993870988, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 6.599277195883642},
    'sf': {'stk__enet__filter__keep': 0.15, 'stk__enet__filter__wins': 6, 'stk__enet__model__enet__alpha': 0.017404995100538886, 'stk__enet__model__enet__l1_ratio': 0.3352843737002149, 'stk__final_estimator__alpha': 0.0889060202841461, 'stk__final_estimator__l1_ratio': 0.3132556552708109, 'stk__hgb__model__hgb__learning_rate': 0.13786633719101948, 'stk__hgb__model__hgb__max_depth': 3, 'stk__pls__model__pls__n_components': 10, 'stk__ridge__model__ridge__alpha': 0.5326262097599814}
}

In [ ]:
data_z['city'].value_counts()

In [ ]:
from sklearn.base           import clone
from sklearn.inspection     import permutation_importance
from sklearn.metrics        import r2_score
import pandas as pd

# ── Assumptions ─────────────────────────────────────────────────────────────
# full_pipe            = Pipeline([('stk', your StackingRegressor pipeline)])
# best_params_by_city  = {'nyc': {...}, 'chicago': {...}, …, 'stl': {...}}
# data_z               = DataFrame with columns = num_features + ['city','trips_per_day']
# num_features         = list of all feature (and lag) column names

city_importances = {}

for city, params in best_params_by_city.items():
    # split out train / test by city
    is_test  = data_z['city'] == city
    is_train = ~is_test

    X_train = data_z.loc[is_train, num_features]
    y_train = data_z.loc[is_train, 'trips_per_day'].values

    X_test  = data_z.loc[is_test,  num_features]
    y_test  = data_z.loc[is_test,  'trips_per_day'].values

    # rebuild & fit the pipeline with that city’s optimal params
    mdl = clone(full_pipe).set_params(**params)
    mdl.fit(X_train, y_train)

    # compute permutation importance on the hold‑out city
    perm = permutation_importance(
        mdl, X_test, y_test,
        scoring    = 'r2',
        n_repeats  = 5,
        random_state = 42,
        n_jobs     = -1
    )

    imp_df = (
        pd.DataFrame({
            'feature': X_test.columns,
            'importance_mean': perm.importances_mean,
            'importance_std' : perm.importances_std
        })
        .sort_values('importance_mean', ascending=False)
        .reset_index(drop=True)
    )

    city_importances[city] = imp_df

# ── Example: print top 10 for St. Louis ──────────────────────────────────────
print("Top 10 features by permutation importance")
print(city_importances['stl'].head(10).to_string(index=False))

# ── (Optional) Average importances across all cities ────────────────────────
all_imp = pd.concat(
    [df.set_index('feature')['importance_mean'] for df in city_importances.values()],
    axis=1
)
all_imp.columns = city_importances.keys()

mean_imp = all_imp.mean(axis=1).sort_values(ascending=False)
print("\nAverage permutation importance across cities (top 10):")
print(mean_imp.head(10).to_string())


In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np

# ---------- A) choose what to analyse ----------
# • entire data  →  global importance
# • STL only     →  influence on STL prediction
#mask = data_z['city'] == 'stl'        # <- change / drop this line for global

X_pi = data_z.loc[:,  num_features]   # raw columns (incl. lags)
y_pi = data_z.loc[:, 'trips_per_day']

# ---------- B) run permutation importance ----------
perm = permutation_importance(
    final_stack,               # <-- your fitted Pipeline
    X_pi, y_pi,
    scoring      = 'r2',
    n_repeats    = 5,
    random_state = 42,
    n_jobs       = -1
)

imp_df = (
    pd.DataFrame({
        'feature' : X_pi.columns,
        'mean'    : perm.importances_mean,
        'std'     : perm.importances_std
    })
    .sort_values('mean', ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 global raw‑feature importances:")
print(imp_df.head(20).to_string(index=False))


In [ ]:
corrs.to_csv('./corr_outputs/all_correlations.csv')

In [ ]:
from pathlib import Path
import pandas as pd

# -----------------------------------------------------------
# Helper: inspect & save correlations for a given feature list
# -----------------------------------------------------------
def inspect_corrs(
        var_list,
        corrs_df,
        out_dir="corr_outputs",
        keep_thresh=KEEP_THRESH,
        wins_thresh=WINS_THRESH,
        sort_by=None    # "mean", "median", or None
    ):
    """
    Show and save per-city Pearson r values for *var_list*,
    plus each variable's win count. Columns sorted alphabetically;
    correlation values rounded to three decimal places.

    Parameters
    ----------
    var_list : list-like of str
        Feature names of interest; must be present in corrs_df.index.
    corrs_df : DataFrame
        Rows = features, columns = cities.
    out_dir : str or Path
        Folder where CSV / TXT files go (created if missing).
    keep_thresh : float
        Absolute-correlation threshold that defines a "win".
    wins_thresh : int
        Unused inside this function but included in the filenames.
    sort_by : {"mean", "median", None}
        If set, rows sorted by that statistic of |r|.

    Returns
    -------
    DataFrame (features × cities) with an extra 'wins' column.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Pull requested rows
    sub = corrs_df.loc[var_list].copy()

    # 2) Compute win counts
    abs_sub = sub.abs()
    sub['wins'] = (abs_sub >= keep_thresh).sum(axis=1)

    # 3) Optional row-sorting by statistic
    if sort_by is not None:
        agg = abs_sub.agg(sort_by, axis=1)
        sub = sub.loc[agg.sort_values(ascending=False).index]

    # 4) Round correlation values to 3 decimals (leave 'wins' intact)
    numeric_cols = [c for c in sub.columns if c != 'wins']
    sub[numeric_cols] = sub[numeric_cols].round(3)

    # 5) Sort columns alphabetically, put 'wins' last
    sorted_cols = sorted(numeric_cols) + ['wins']
    sub = sub[sorted_cols]

    # 6) Show in console
    with pd.option_context("display.max_columns", None):
        print("\nPer-city Pearson r (plus win counts):")
        print(sub)

    # 7) Build filenames
    stem = f"keep{keep_thresh:g}_wins{wins_thresh}"
    csv_path  = out_dir / f"wide_feature_city_correlations_{stem}_2010_sorted_lags.csv"
    vars_path = out_dir / f"wide_selected_variables_{stem}_2010_sorted_lags.txt"

    # 8) Save outputs
    sub.to_csv(csv_path, index=True)
    with open(vars_path, "w") as f:
        f.writelines(f"{v}\n" for v in var_list)

    print(f"\n→ Table saved to {csv_path}")
    print(f"→ Variable list saved to {vars_path}")

    return sub

# ---------------- EXAMPLE USAGE -----------------
sub_df = inspect_corrs(
   var_list=selected,
   corrs_df=corrs,
   out_dir="corr_outputs",
   sort_by="mean"
)

In [ ]:
# Lasso filter
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Define X and y
X = data_z[selected]   # or data[selected_consistent]
y = data_z['trips_per_day']

# 2. Build a pipeline: standardize then run Lasso with 5-fold CV
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=42, n_jobs=-1))
])

# 3. Fit the model
pipe.fit(X, y)

# 4. Extract non-zero coefficients
coef = pd.Series(pipe.named_steps['lasso'].coef_, index=X.columns)
selected_lasso = coef[coef != 0]

# 5. Output the results
print(f"Train R^2: {pipe.score(X, y):.3f}\n")
print("Features selected by Lasso (non-zero coefficients):")
print(selected_lasso.sort_values(ascending=False))


In [ ]:
print(f'{len(selected_lasso)} Features after Coefficient, Collinear, and Lasso selection')

In [ ]:
selected_lasso.sort_values()

In [ ]:
# Lag making
from sklearn.neighbors import NearestNeighbors

# 2.1 Pick the features you want to lag (your winnowed set)
to_lag = selected_lasso.index.tolist()

# 2.2 Build the pooled k-NN on centroids
coords = data_z[['centroid_x','centroid_y']].values
nbrs  = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs.kneighbors(coords)

# 2.3 Create lag columns
for col in to_lag:
    lag_col = f"{col}_lag5"
    arr     = data_z[col].values
    data_z[lag_col] = [
        arr[neighbors[1:]].mean() 
        for neighbors in idxs
    ]

# 2.4 Extend your final feature list
lag_cols = [f"{c}_lag5" for c in to_lag]
# ensure selected_final is a plain list of column-names
selected_list = list(selected_lasso)

# lag_cols is already a Python list of strings:
lag_cols = [f"{c}_lag5" for c in to_lag]

# now a clean list concatenation
selected_with_lags = selected_list + lag_cols

# 1. Base features come from the index of your selected_lasso Series
base_features = selected_lasso.index.tolist()

# 2. Create the lag‐feature names
lag_cols = [f"{feat}_lag5" for feat in base_features]

# 3. Concatenate two pure Python lists
selected_with_lags = base_features + lag_cols

# Quick sanity check: only strings, no numbers
print(any(not isinstance(x, str) for x in selected_with_lags))  # should be False
print("First 5 features:", selected_with_lags[:5])
print("Last 5 lags:    ", selected_with_lags[-5:])
print('Final feature count before fit: ', len(selected_with_lags))

from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh

# # 1) Build the k‐NN adjacency graph (reuse your coords array)
# coords = data_z[['centroid_x','centroid_y']].values
# A      = kneighbors_graph(coords, n_neighbors=6,
#                           mode='connectivity', include_self=False)

# # 2) Compute the (unnormalized or normalized) Laplacian
# #    normed=True often gives nicer spectral embeddings.
# L = laplacian(A, normed=True)

# # 3) Compute the first k+1 eigenpairs of L (smallest eigenvalues)
# #    The zero eigenvalue (constant eigenvector) lives at index 0.
# k = 5
# # which='SM' → smallest magnitude eigenvalues
# eigvals, eigvecs = eigsh(L, k=k+1, which='SM')  

# # 4) Discard the trivial first eigenvector, keep the next k
# #    and add them to your DataFrame
# for i in range(1, k+1):
#     data_z[f'spec_{i}'] = eigvecs[:, i]

# print(f"Added spec_1…spec_{k} to data_z; new shape = {data_z.shape}")

# # 5) Now include those in your feature lists before lags / pipelines:
# #    e.g. if you had base_enet = sel4_enet.tolist(), do:
# #base_enet += [f'spec_{i}' for i in range(1, k+1)]
# #    then compute your lag cols for those as well (if you like),
# #    and proceed to make_pipe(...) exactly as before.



In [ ]:
print(data_z['trips_per_day'])

In [ ]:
# ------------------------------------------------------------------
# 0.  Setup  (requires scikit-learn ≥ 1.1)
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# ----------------------------------------------------
# 1.  Feature matrix X, target y, and grouping by city
# ----------------------------------------------------
#X   = data_z[selected_columns]             # your 43 vetted predictors
X   = data_z[selected_with_lags]             # your 43 vetted predictors

y   = data_z['trips_per_day']
#y   = np.log1p(data_z['trips_per_day'].values)

grp = data_z['city'].values              # ensures each CV fold is a whole city

gkf = GroupKFold(n_splits=len(np.unique(grp)))  # 11 folds

# ----------------------------------------------------
# 2.  Define three candidate models
# ----------------------------------------------------
# ElasticNet best params: {'enet__alpha': 0.046415888336127774, 'enet__l1_ratio': 0.2, 'enet__max_iter': 1000, 'enet__positive': False, 'enet__tol': 1e-05}
# ElasticNet best params: {'keep': 0.1, 'wins': 9, 'alpha': 0.01, 'l1_ratio': 0.5, 'r2': 0.33728442700537814, 'corr_feats': 62, 'lasso_feats': 23}
# ElasticNet best params w log{'keep': 0.1, 'wins': 5, 'alpha': 0.001, 'l1_ratio': 1.0, 'r2': 0.4140527445186853, 'corr_feats': 121, 'lasso_feats': 53}

# RandomForest best params w log {'hgb_keep': 0.1, 'hgb_wins': 5, 'learning_rate': 0.1, 'max_depth': 5, 'max_iter': 100, 'r2': 0.3782833030650996, 'corr_feats': 121, 'lasso_feats': 53}

models = {
    "ElasticNet": Pipeline([
#    ('scaler', StandardScaler()),
    ('enet', ElasticNet(
        alpha     = 0.001,
        l1_ratio  = 1.0,
        max_iter  = 100000,
        random_state=42
    ))
    ]),

    # "RandomForest": RandomForestRegressor(
    #     n_estimators=400, max_depth=None, n_jobs=-1, random_state=42),
    # HGB best params: {'l2_regularization': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'min_samples_leaf': 50}
    # "HGB": HistGradientBoostingRegressor(
    #     learning_rate=0.01, max_depth=5, l2_regularization=1.0, max_iter = 2000,
    #     max_depth=5,
    #     min_samples_leaf = 50, random_state=42), # .324 w 168 feats

    #  HGB best params: {'l2_regularization': 1.0, 'learning_rate': 0.05, 'max_depth': 10, 'min_samples_leaf': 20} .307 w 63 feats
#     HistGradientBoostingRegressor(early_stopping=True, l2_regularization=1.0,
#                               learning_rate=0.05, max_depth=10, max_iter=2000,
#                               n_iter_no_change=50, random_state=42, tol=0.0001)
# {'categorical_features': 'warn', 'early_stopping': True, 'interaction_cst': None, 'l2_regularization': 1.0, 'learning_rate': 0.05, 'loss': 'squared_error', 'max_bins': 255, 'max_depth': 10, 'max_features': 1.0, 'max_iter': 2000, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'monotonic_cst': None, 'n_iter_no_change': 50, 'quantile': None, 'random_state': 42, 'scoring': 'loss', 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
#     
#     "HGB": HistGradientBoostingRegressor(
#     learning_rate=0.05,
#     max_depth=10,
#     min_samples_leaf=20,
#     l2_regularization=1.0,
#     max_iter=2000,
#     early_stopping=True,
#     validation_fraction=0.1,
#     n_iter_no_change=50,
#     tol=1e-4,
#     random_state=42
# )

}


scoring = ('r2', 'neg_root_mean_squared_error')
cv_results = {}

# ----------------------------------------------------
# 3.  Cross-validate each model (group-aware)
# ----------------------------------------------------
for name, mdl in models.items():
    res = cross_validate(mdl, X, y, cv=gkf, scoring=scoring,
                         return_train_score=False, groups=grp, n_jobs=-1)
    cv_results[name] = {k: v.mean() for k, v in res.items()}
    print(f"{name:12s}  mean R² = {res['test_r2'].mean():.3f}   "
          f"RMSE = {-res['test_neg_root_mean_squared_error'].mean():.1f}")

# ----------------------------------------------------
# 4.  Pick the best model by mean R²
# ----------------------------------------------------
best_name = max(cv_results, key=lambda n: cv_results[n]['test_r2'])
best_model = models[best_name].fit(X, y)     # refit on full data
print(f"\n✅  Best overall model: {best_name}")

# # ----------------------------------------------------
# # 5.  Extract & show the TOP-5 drivers
# # ----------------------------------------------------
# if best_name == "ElasticNet":
#     coefs = pd.Series(
#         best_model.named_steps['enet'].coef_,
#         index=X.columns
#     ).abs().sort_values(ascending=False)
#     top5 = coefs.head(5)
#     print("\nTop-5 variables (|standardised β|):")
#     print(top5)

# else:  # tree model → permutation importance
#     perm = permutation_importance(best_model, X, y,
#                                   n_repeats=30, random_state=0, n_jobs=-1,
#                                   scoring='r2')
#     importances = pd.Series(perm.importances_mean, index=X.columns)
#     top5 = importances.sort_values(ascending=False).head(5)
#     print("\nTop-5 variables (permutation importance decrease in R²):")
#     print(top5.round(4))

# You now have:
#   • cv_results  → dict of mean CV metrics
#   • best_model  → final fitted model
#   • top5        → the five variables to feature in your report


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model      import ElasticNet, ElasticNetCV
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler
from sklearn.model_selection   import GroupKFold, cross_val_score
from sklearn.metrics           import r2_score

# 1) Hyper‑parameter grids
keep_grid    = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid    = [5, 6, 7, 8, 9, 10]
# for the final model
alpha_grid   = [0.001, 0.01, 0.1, 1.0]
l1_grid      = [0.2, 0.5, 0.8, 1.0]
# for the ElasticNetCV *filter*
sel_alpha_grid = [0.001, 0.01, 0.1, 1.0]
sel_l1_grid    = [0.2, 0.5, 0.8, 1.0]

groups    = data_z['city'].values
cities    = np.unique(groups)
outer_cv  = GroupKFold(n_splits=len(cities))
y_all     = data_z['trips_per_day'].values

outer_scores = []

for train_idx, test_idx in outer_cv.split(data_z, groups=groups):
    train = data_z.iloc[train_idx]
    test  = data_z.iloc[test_idx]
    y_train = train['trips_per_day'].values
    y_test  = test ['trips_per_day'].values

    # inner LOCO over the *training* cities
    inner_groups = train['city'].values
    inner_cv = GroupKFold(n_splits=len(np.unique(inner_groups)))

    best_inner_score = -np.inf
    best_cfg = None

    # ----- INNER GRID SEARCH -----
    for keep in keep_grid:
        for wins in wins_grid:
            # (1) corr→collinear→same‑sign filter on TRAIN only
            feats1 = filter_features(
                corrs, train,
                keep_thresh     = keep,
                wins_thresh     = wins,
                colinear_thresh = REDUNDANCY_R,
                same_sign_frac  = SAME_SIGN_FRAC
            )
            if len(feats1) < 2:
                continue

            X1 = train[feats1].values

            # (2) ElasticNetCV feature‑selection on TRAIN
            enet_cv = Pipeline([
                ('scale', StandardScaler()),
                ('sel',   ElasticNetCV(
                             alphas      = sel_alpha_grid,
                             l1_ratio    = sel_l1_grid,
                             cv          = 5,
                             random_state= 42,
                             n_jobs      = -1
                         ))
            ]).fit(X1, y_train).named_steps['sel']

            coefs = pd.Series(enet_cv.coef_, index=feats1)
            feats2 = coefs[coefs != 0].index.tolist()
            if len(feats2) < 2:
                continue

            X2 = train[feats2].values

            # (3) final ElasticNet grid on the 2×‑filtered set
            for alpha in alpha_grid:
                for l1 in l1_grid:
                    pipe = Pipeline([
                        ('scale', StandardScaler()),
                        ('enet',  ElasticNet(
                                     alpha       = alpha,
                                     l1_ratio    = l1,
                                     max_iter    = 10_000,
                                     random_state= 42
                                 ))
                    ])
                    # LOCO R² *within* training cities
                    inner_scores = cross_val_score(
                        pipe, X2, y_train,
                        cv=inner_cv.split(train, groups=inner_groups),
                        scoring='r2',
                        n_jobs=-1
                    )
                    mean_inner = inner_scores.mean()

                    if mean_inner > best_inner_score:
                        best_inner_score = mean_inner
                        best_cfg = {
                            'keep' : keep,
                            'wins' : wins,
                            'sel_alpha': enet_cv.alpha_,
                            'sel_l1'   : enet_cv.l1_ratio_,
                            'final_alpha': alpha,
                            'final_l1'   : l1,
                            'feats2'     : feats2
                        }

    # end inner grid
    if best_cfg is None:
        raise RuntimeError("No valid config in inner loop")

    # ----- RE‑FIT on full TRAIN set using best inner config -----
    feats2     = best_cfg['feats2']
    X_train_best = train[feats2].values
    final_model = Pipeline([
        ('scale', StandardScaler()),
        ('enet',  ElasticNet(
                     alpha       = best_cfg['final_alpha'],
                     l1_ratio    = best_cfg['final_l1'],
                     max_iter    = 10_000,
                     random_state= 42
                 ))
    ])
    final_model.fit(X_train_best, y_train)

    # ----- EVALUATE on held‑out CITY -----
    X_test_best = test[feats2].values
    y_pred      = final_model.predict(X_test_best)
    outer_scores.append(r2_score(y_test, y_pred))

# summarize true nested LOCO R²
outer_scores = np.array(outer_scores)
print(
  f"Nested LOCO R² = "
  f"{outer_scores.mean():.4f} ± {outer_scores.std(ddof=1):.4f}"
)

In [ ]:
# Enet everything tuning with enet filter
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline       import Pipeline
from sklearn.preprocessing  import StandardScaler
from sklearn.linear_model   import ElasticNet, ElasticNetCV

# ── Assumes these are defined:
#    data_z, corrs, filter_features,
#    REDUNDANCY_R, SAME_SIGN_FRAC
groups    = data_z['city'].values
outer_cv  = GroupKFold(n_splits=len(np.unique(groups)))
y         = data_z['trips_per_day'].values

# your parameter grids
keep_grid  = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid  = [5, 6, 7, 8, 9, 10, 11]

# we'll use these both for selection and final tuning
alpha_grid = [0.001, 0.01, 0.1, 1.0]
l1_grid    = [0.2, 0.5, 0.8, 1.0]

results    = []
best_score = -np.inf
enet_best_config  = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")
        # 1–3) corr + collinear + same-sign
        feats = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
        print(f" after corr+/col+/same-sign: {len(feats)} features")
        if len(feats) < 2:
            print("  → too few after filters, skipping")
            continue

        # build X for selection
        X_prefilt = data_z[feats].values

        # 4) ElasticNetCV filter
        sel = Pipeline([
            ('scaler',   StandardScaler()),
            ('enet_sel', ElasticNetCV(
                l1_ratio    = l1_grid,
                alphas      = alpha_grid,
                cv          = 5,
                random_state=42,
                n_jobs      = -1
            ))
        ]).fit(X_prefilt, y)

        coefs    = pd.Series(
            sel.named_steps['enet_sel'].coef_,
            index=feats
        )
        feats2   = coefs[coefs != 0].index.tolist()
        print(f" after ElasticNetCV filter: {len(feats2)} features")
        if len(feats2) < 2:
            print("  → too few after ENetCV, skipping")
            continue

        # final matrix for grid‐search
        X_final = data_z[feats2].values

        # 5) now loop over ElasticNet hyperparams
        for alpha in alpha_grid:
            for l1 in l1_grid:
                pipe = Pipeline([
                    ('scaler', StandardScaler()),
                    ('enet',   ElasticNet(
                        alpha       = alpha,
                        l1_ratio    = l1,
                        max_iter    = 10_000,
                        random_state=42
                    ))
                ])
                scores = cross_val_score(
                    pipe, X_final, y,
                    cv=outer_cv.split(data_z, groups=groups),
                    scoring='r2', n_jobs=-1
                )
                mean_r2 = scores.mean()
                print(f"   α={alpha:.3f}, l1={l1:.1f} → R²={mean_r2:.4f}")

                results.append({
                    'keep'      : keep,
                    'wins'      : wins,
                    'alpha'     : alpha,
                    'l1_ratio'  : l1,
                    'n_prefilt' : len(feats),
                    'n_enetsel' : len(feats2),
                    'r2'        : mean_r2
                })
                if mean_r2 > best_score:
                    best_score = mean_r2
                    enet_best_config  = results[-1]

# summarize
print("\n=== Best config ===")
print(enet_best_config)

df = pd.DataFrame(results)
print(df.sort_values('r2', ascending=False).head(10))


In [ ]:
# Elasticnet everything tuning #nofilter
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ElasticNet
from sklearn.neighbors import NearestNeighbors

# ── Assumes data_z (z-scored DataFrame with centroid_x, centroid_y) and corrs (feature-city correlations) are in scope

groups = data_z['city'].values
outer_cv = GroupKFold(n_splits=len(np.unique(groups)))
y = data_z['trips_per_day']
#y   = np.log1p(data_z['trips_per_day'].values)


# Precompute 5-NN neighbor indices on centroids
# coords = data_z[['centroid_x', 'centroid_y']].values
# nbrs_model = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
# _, idxs = nbrs_model.kneighbors(coords)

# define grids
keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [5, 6, 7, 8, 9, 10, 11]
alpha_grid  = [0.001, 0.01, 0.1, 1,0]
l1_grid     = [0.2, 0.5, 0.8, 1.0]


results = []
best_score  = -np.inf
enet_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        # … your corr + wins filtering, collinearity, same-sign, LassoCV as before …
        # assume you arrive at X_model, sel4, etc.
        # (see your original code for steps 1–5)

        # Only proceed if you still have enough features
        print(f"\n=== keep={keep}, wins={wins} ===")
        corr_selected = filter_features(
            corrs, data_z,
            keep_thresh=keep,
            wins_thresh=wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        if len(corr_selected) < 2:
            print("  → too few features after corr+wins, skipping.")
            continue
        
        X_model = data_z[corr_selected]
        # # 4) LassoCV filter
        # X_lasso = data_z[corr_selected].values
        # lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        # coefs = pd.Series(lasso.coef_, index=corr_selected)
        # lasso_selected = coefs[coefs != 0].index.tolist()
        # print(" after LassoCV:", len(lasso_selected))

        # # if len(lasso_selected) < 5:
        # #     continue

        # # # 5) build lag features
        # X_model = data_z[lasso_selected].values

        # if len(lasso_selected) < 5:
        #     print("  → fewer than 5 Lasso-selected features, skipping ElasticNet.")
        #     continue
        
        # lag_arrays = []
        # for col in lasso_selected:
        #     arr = data_z[col].values
        #     lag_arrays.append(np.array([arr[n[1:]].mean() for n in idxs]))
        # X_lag = np.column_stack(lag_arrays)
        # X_model = np.hstack([X_base, X_lag])

        if X_model.shape[1] < 2:
            print("  → too few total features for ElasticNet, skipping.")
            continue

        # 6) now loop over ElasticNet hyperparams
        for alpha in alpha_grid:
            for l1_ratio in l1_grid:
                pipe = Pipeline([
                    ('scaler', StandardScaler()),
                    ('enet', ElasticNet(
                        alpha=alpha,
                        l1_ratio=l1_ratio,
                        random_state=42,
                        max_iter=10000
                    ))
                ])
                scores = cross_val_score(
                    pipe, X_model, y,
                    cv=outer_cv.split(data_z, groups=groups),
                    scoring='r2', n_jobs=-1
                )
                mean_r2 = scores.mean()
                print(f" keep={keep}, wins={wins}, α={alpha}, l1={l1_ratio} → R²={mean_r2:.4f}")

                results.append({
                    'keep': keep,
                    'wins': wins,
                    'alpha': alpha,
                    'l1_ratio': l1_ratio,
                    'r2': mean_r2,
                    # optionally track intermediate sizes:
                    'corr_feats': len(corr_selected),
                    'lasso_feats': len(corr_selected),
                })

                if mean_r2 > best_score:
                    best_score  = mean_r2
                    enet_best_config = results[-1]

# summarize
print("\n=== Best config ===")
print(enet_best_config)

df = pd.DataFrame(results)
print(df.sort_values('r2', ascending=False).head())


In [ ]:
# Elasticnet everything tuning with lag building
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ElasticNet
from sklearn.neighbors import NearestNeighbors

# ── Assumes data_z (z-scored DataFrame with centroid_x, centroid_y) and corrs (feature-city correlations) are in scope

groups = data_z['city'].values
outer_cv = GroupKFold(n_splits=len(np.unique(groups)))
y = data_z['trips_per_day']
#y   = np.log1p(data_z['trips_per_day'].values)


# Precompute 5-NN neighbor indices on centroids
coords = data_z[['centroid_x', 'centroid_y']].values
nbrs_model = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs_model.kneighbors(coords)

# define grids
keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [5, 6, 7, 8, 9, 10, 11]
alpha_grid  = [0.001, 0.01, 0.1, 1,0]
l1_grid     = [0.2, 0.5, 0.8, 1.0]


results = []
best_score  = -np.inf
enet_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        # … your corr + wins filtering, collinearity, same-sign, LassoCV as before …
        # assume you arrive at X_model, sel4, etc.
        # (see your original code for steps 1–5)

        # Only proceed if you still have enough features
        print(f"\n=== keep={keep}, wins={wins} ===")
        corr_selected = filter_features(
            corrs, data_z,
            keep_thresh=keep,
            wins_thresh=wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        if len(corr_selected) < 2:
            print("  → too few features after corr+wins, skipping.")
            continue

        # 4) LassoCV filter
        X_lasso = data_z[corr_selected].values
        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        coefs = pd.Series(lasso.coef_, index=corr_selected)
        lasso_selected = coefs[coefs != 0].index.tolist()
        print(" after LassoCV:", len(lasso_selected))

        if len(lasso_selected) < 5:
            continue

        # 5) build lag features
        X_base = data_z[lasso_selected].values

        if len(lasso_selected) < 5:
            print("  → fewer than 5 Lasso-selected features, skipping ElasticNet.")
            continue
        
        lag_arrays = []
        for col in lasso_selected:
            arr = data_z[col].values
            lag_arrays.append(np.array([arr[n[1:]].mean() for n in idxs]))
        X_lag = np.column_stack(lag_arrays)
        X_model = np.hstack([X_base, X_lag])

        if X_model.shape[1] < 2:
            print("  → too few total features for ElasticNet, skipping.")
            continue

        # 6) now loop over ElasticNet hyperparams
        for alpha in alpha_grid:
            for l1_ratio in l1_grid:
                pipe = Pipeline([
                    ('scaler', StandardScaler()),
                    ('enet', ElasticNet(
                        alpha=alpha,
                        l1_ratio=l1_ratio,
                        random_state=42,
                        max_iter=10000
                    ))
                ])
                scores = cross_val_score(
                    pipe, X_model, y,
                    cv=outer_cv.split(data_z, groups=groups),
                    scoring='r2', n_jobs=-1
                )
                mean_r2 = scores.mean()
                print(f" keep={keep}, wins={wins}, α={alpha}, l1={l1_ratio} → R²={mean_r2:.4f}")

                results.append({
                    'keep': keep,
                    'wins': wins,
                    'alpha': alpha,
                    'l1_ratio': l1_ratio,
                    'r2': mean_r2,
                    # optionally track intermediate sizes:
                    'corr_feats': len(corr_selected),
                    'lasso_feats': len(lasso_selected),
                })

                if mean_r2 > best_score:
                    best_score  = mean_r2
                    enet_best_config = results[-1]

# summarize
print("\n=== Best config ===")
print(enet_best_config)

df = pd.DataFrame(results)
print(df.sort_values('r2', ascending=False).head())


In [ ]:
#{'keep': 0.1, 'wins': 6, 'alpha': 0.01, 'l1_ratio': 0.8, 'r2': 0.35038782912780225, 'corr_feats': 118, 'lasso_feats': 44}

In [ ]:
# Old ElasticNet keep/wins correlation tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.neighbors import NearestNeighbors

# ── Assumes data_z (z-scored DataFrame with centroid_x, centroid_y) and corrs (feature-city correlations) are in scope

groups = data_z['city'].values
outer_cv = GroupKFold(n_splits=len(np.unique(groups)))
y = data_z['trips_per_day'].values

# Precompute 5-NN neighbor indices on centroids
coords = data_z[['centroid_x', 'centroid_y']].values
nbrs_model = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs_model.kneighbors(coords)

results = []
best_score = -np.inf
best_config = None

for keep in [0.10, 0.15, 0.20, 0.25, 0.30]:
    for wins in [5, 6, 7, 8, 9, 10]:
        print(f"\n=== keep={keep}, wins={wins} ===")
        # 1) corr + wins filter
        abs_corrs = corrs.abs()
        summary = (
            abs_corrs
              .agg(['mean','median'], axis=1)
              .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
              .assign(wins=(abs_corrs >= keep).sum(axis=1))
        )
        selected = summary.query("mean_abs_r>=@keep and wins>=@wins").index.tolist()
        print(" after corr filter:", len(selected))

        # 2) drop collinear pairs
        upper = data_z[selected].corr().abs().where(
            np.triu(np.ones((len(selected), len(selected)), dtype=bool), k=1)
        )
        to_drop = [c for c in upper.columns if any(upper[c] > 0.90)]
        sel2 = [c for c in selected if c not in to_drop]
        print(" after collinearity:", len(sel2))

        if len(sel2) < 5:
            continue

        # 3) same-sign filter
        corrs_sel = corrs.loc[sel2]
        signs_sel = corrs_sel.apply(np.sign)
        modal = signs_sel.mode(axis=1)[0]
        sign_frac = signs_sel.eq(modal, axis=0).sum(axis=1) / signs_sel.shape[1]
        sel3 = sign_frac[sign_frac == 1.0].index.tolist()
        print(" after same-sign:", len(sel3))

        if len(sel3) < 5:
            continue

        # 4) LassoCV filter
        X_lasso = data_z[sel3].values
        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        coefs = pd.Series(lasso.coef_, index=sel3)
        sel4 = coefs[coefs != 0].index.tolist()
        print(" after LassoCV:", len(sel4))

        if len(sel4) < 5:
            continue

        # 5) build lag features
        X_base = data_z[sel4].values
        lag_arrays = []
        for col in sel4:
            arr = data_z[col].values
            lag_arrays.append(np.array([arr[n[1:]].mean() for n in idxs]))
        X_lag = np.column_stack(lag_arrays)
        X_model = np.hstack([X_base, X_lag])

        # 6) ElasticNet LOCO R²
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('enet', ElasticNetCV(l1_ratio=[.2,.5,.8,1], cv=5, random_state=42, n_jobs=-1))
        ])
        scores = cross_val_score(pipe, X_model, y,
                                 cv=outer_cv.split(data_z, groups=groups),
                                 scoring='r2', n_jobs=-1)
        mean_r2 = scores.mean()
        print(" ElasticNet LOCO R²:", mean_r2)

        results.append({
            'keep': keep, 'wins': wins,
            'corr_feats': len(selected),
            'collinear_removed': len(to_drop),
            'same_sign': len(sel3),
            'lasso_feats': len(sel4),
            'r2': mean_r2
        })
        if mean_r2 > best_score:
            best_score = mean_r2
            best_config = results[-1]

print("\n=== Best config ===")
print(best_config)
pd.DataFrame(results)


In [ ]:
# RandomForest keep/wins correlation tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.neighbors import NearestNeighbors

# Assumes data_z (z-scored DataFrame) and corrs (feature-city correlations DataFrame) exist

groups = data_z['city'].values
outer_cv = GroupKFold(n_splits=len(np.unique(groups)))

# Precompute k-NN neighbor indices on (centroid_x, centroid_y)
coords = data_z[['centroid_x', 'centroid_y']].values
nbrs_model = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs_model.kneighbors(coords)

results = []
best_score = -np.inf
best_config = None

y = data_z['trips_per_day'].values

for keep in [0.10, 0.15, 0.20, 0.25, 0.30]:
    for wins in [7, 8, 9, 10]:
        print(f"\n=== Testing keep_thresh={keep}, wins_thresh={wins} ===")
        # 1) Correlation strength & wins filter
        abs_corrs = corrs.abs()
        summary = (
            abs_corrs
              .agg(['mean', 'median'], axis=1)
              .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
              .assign(wins=(abs_corrs >= keep).sum(axis=1))
        )
        selected = summary.query("mean_abs_r >= @keep and wins >= @wins").index.tolist()
        print(f"After corr filter: {len(selected)} features")

        # 2) Drop highly collinear pairs (|r| > 0.90)
        if not selected:
            continue
        upper = (
            data_z[selected]
              .corr()
              .abs()
              .where(np.triu(np.ones((len(selected), len(selected)), dtype=bool), k=1))
        )
        to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
        selected2 = [c for c in selected if c not in to_drop]
        print(f"After collinearity filter: {len(selected2)} features")

        if len(selected2) < 5:
            continue

        # 3) Same-sign across cities filter
        corrs_sel = corrs.loc[selected2]
        signs_sel = corrs_sel.apply(np.sign)
        modal_sign = signs_sel.mode(axis=1)[0]
        sign_frac = (signs_sel.eq(modal_sign, axis=0).sum(axis=1) / signs_sel.shape[1])
        selected3 = sign_frac[sign_frac == 1.0].index.tolist()
        print(f"After same-sign filter: {len(selected3)} features")

        if len(selected3) < 5:
            continue

        # 4) LassoCV filter (drop zero coefficients)
        X_lasso = data_z[selected3].values
        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        coefs = pd.Series(lasso.coef_, index=selected3)
        selected4 = coefs[coefs != 0].index.tolist()
        print(f"After LassoCV filter: {len(selected4)} features")

        if len(selected4) < 5:
            continue

        # 5) Build spatial lag features for selected4
        X_base = data_z[selected4].values
        lag_arrays = []
        for col in selected4:
            arr = data_z[col].values
            lag = np.array([arr[neighbors[1:]].mean() for neighbors in idxs])
            lag_arrays.append(lag)
        X_lag = np.column_stack(lag_arrays)
        # Combine base + lag features
        X_rf = np.hstack([X_base, X_lag])

        # 6) Evaluate RandomForest LOCO R²
        rf = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
        scores = cross_val_score(
            rf, X_rf, y,
            cv=outer_cv.split(data_z, groups=groups),
            scoring='r2', n_jobs=-1
        )
        mean_r2 = scores.mean()
        print(f"RandomForest LOCO R² with lags: {mean_r2:.3f}")

        # Record results
        results.append({
            'keep_thresh': keep,
            'wins_thresh': wins,
            'n_corr_feats': len(selected),
            'n_collinear_removed': len(to_drop),
            'n_same_sign': len(selected3),
            'n_lasso_feats': len(selected4),
            'r2_with_lags': mean_r2
        })
        if mean_r2 > best_score:
            best_score = mean_r2
            best_config = results[-1]

# Summarize best configuration
print("\n=== Best configuration ===")
print(best_config)

# Full grid results
pd.DataFrame(results)


In [ ]:
# RandoForest Pearson/Spearman correlation tuning
# -------------------------------------------------------------------
# Combined Pearson + Spearman filter grid search for RandomForest LOCO R²
# -------------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.neighbors import NearestNeighbors

# Assumes:
#   - data_z: z-scored DataFrame with 'city', 'trips_per_day', 'centroid_x', 'centroid_y'
#   - num_features: list of numeric predictor names in data_z

# Precompute correlation matrices
corrs = {}
corrs['pearson'] = pd.DataFrame({
    city: grp[num_features].corrwith(grp['trips_per_day'])
    for city, grp in data_z.groupby('city')
}).astype(float)

corrs['spearman'] = pd.DataFrame({
    city: grp[num_features]
             .apply(lambda col: col.corr(grp['trips_per_day'], method='spearman'))
    for city, grp in data_z.groupby('city')
}).astype(float)

# CV setup
groups    = data_z['city'].values
outer_cv  = GroupKFold(n_splits=len(np.unique(groups)))
coords    = data_z[['centroid_x', 'centroid_y']].values
nbrs      = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs   = nbrs.kneighbors(coords)
y         = data_z['trips_per_day'].values

# Threshold grids
keep_p_list = [0.20, 0.25]
wins_p_list = [8, 9, 10]
keep_s_list = [0.20, 0.25]
wins_s_list = [8, 9, 10]

results = []
best_score = -np.inf
best_cfg = None

for keep_p in keep_p_list:
    for wins_p in wins_p_list:
        # Pearson selection
        abs_p = corrs['pearson'].abs()
        sum_p = (
        abs_p
        .agg(['mean'], axis=1)
        .rename(columns={'mean':'mean_abs_r'})   # ← rename the column, not the index
        )
        sum_p['wins'] = (abs_p >= keep_p).sum(axis=1)
        sel_p = sum_p.query("mean_abs_r >= @keep_p and wins >= @wins_p").index.tolist()

        for keep_s in keep_s_list:
            for wins_s in wins_s_list:
                # Spearman selection
                abs_s = corrs['spearman'].abs()
                sum_s = (
                    abs_s
                    .agg(['mean'], axis=1)
                    .rename(columns={'mean':'mean_abs_r'})
                )
                sum_s['wins'] = (abs_s >= keep_s).sum(axis=1)
                sel_s = sum_s.query("mean_abs_r >= @keep_s and wins >= @wins_s").index.tolist()

                # Union of both filters
                selected = list(set(sel_p) | set(sel_s))
                if len(selected) < 5:
                    continue

                # Drop collinear
                upper = data_z[selected].corr().abs().where(np.triu(np.ones((len(selected),)*2), k=1).astype(bool))
                to_drop = [c for c in upper.columns if any(upper[c] > 0.9)]
                selected2 = [c for c in selected if c not in to_drop]
                if len(selected2) < 5:
                    continue

                # Same-sign filter on Pearson *AND* Spearman
                # require sign consistency in both matrices
                def same_sign_filter(mat, feats):
                    signs = mat.loc[feats].apply(np.sign)
                    modal = signs.mode(axis=1)[0]
                    frac  = (signs.eq(modal, axis=0).sum(axis=1) / signs.shape[1])
                    return frac[frac == 1.0].index.tolist()

                ss_p = same_sign_filter(corrs['pearson'], selected2)
                ss_s = same_sign_filter(corrs['spearman'], selected2)
                selected3 = list(set(ss_p) & set(ss_s))
                if len(selected3) < 5:
                    continue

                # LassoCV reduction
                Xlasso = data_z[selected3].values
                lasso  = LassoCV(cv=5, random_state=0).fit(Xlasso, y)
                coef   = pd.Series(lasso.coef_, index=selected3)
                selected4 = coef[coef != 0].index.tolist()
                if len(selected4) < 5:
                    continue

                # Build spatial lags
                X_base = data_z[selected4].values
                lags   = [np.array([data_z[col].values[nbr[1:]].mean() for nbr in idxs])
                          for col in selected4]
                X_lag  = np.column_stack(lags)
                X_rf   = np.hstack([X_base, X_lag])

                rf     = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
                scores = cross_val_score(
                    rf,
                    X_rf,
                    y,
                    cv=outer_cv,      # pass the CV object directly
                    groups=groups,
                    scoring='r2',
                    n_jobs=-1
                )
                mean_r2 = scores.mean()

                # print after every run
                print(f" keep_p={keep_p:0.2f}, wins_p={wins_p}, "
                    f"keep_s={keep_s:0.2f}, wins_s={wins_s}, "
                    f"n_feat={len(selected4)}, R²={mean_r2:.4f}")

                # now record the true mean_r2
                results.append({
                    'keep_p': keep_p, 'wins_p': wins_p,
                    'keep_s': keep_s, 'wins_s': wins_s,
                    'n_feat': len(selected4),
                    'r2': mean_r2
                })
                if mean_r2 > best_score:
                    best_score = mean_r2
                    best_cfg = results[-1]

print("Best config:", best_cfg)
pd.DataFrame(results).head()


In [ ]:
from functools import lru_cache
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingGridSearchCV

@lru_cache(maxsize=None)
def get_feats1(keep, wins):
    return filter_features(corrs, data_z, keep, wins, REDUNDANCY_R, SAME_SIGN_FRAC)

# predefine pipelines
enet_sel_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('sel',   ElasticNetCV(
                alphas    = sel_alpha_grid,
                l1_ratio  = sel_l1_grid,
                cv        = 5,
                random_state=42,
                n_jobs    = 1   # don't parallelize inside
             ))
])

final_enet_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('enet',  ElasticNet(max_iter=10_000, random_state=42))
])

param_grid = {
    'enet__alpha':     alpha_grid,
    'enet__l1_ratio':  l1_grid
}

halving = HalvingGridSearchCV(
    final_enet_pipe,
    param_grid   = param_grid,
    factor       = 3,
    cv           = inner_cv,
    scoring      = 'r2',
    n_jobs        = 1,  # only one at a time
    verbose      = 1
)

for train_idx, test_idx in outer_cv.split(data_z, groups=groups):
    train = data_z.iloc[train_idx]
    test  = data_z.iloc[test_idx]
    y_train = train['trips_per_day'].values
    y_test  = test ['trips_per_day'].values

    best_inner = -np.inf
    best_cfg = None

    for keep in keep_grid:
      for wins in wins_grid:
        feats1 = get_feats1(keep, wins)
        if len(feats1) < 2: continue
        X1 = train[feats1].values

        # 1) run ElasticNetCV filter ONCE per feats1
        enet_sel_pipe.fit(X1, y_train)
        sel = enet_sel_pipe.named_steps['sel']
        feats2 = [
          f for f,coef in zip(feats1, sel.coef_)
          if coef != 0
        ]
        if len(feats2) < 2: continue

        X2 = train[feats2].values

        # 2) Halving search over final enet hyperparams
        halving.fit(X2, y_train)
        mean_inner = halving.best_score_

        if mean_inner > best_inner:
          best_inner = mean_inner
          best_cfg = {
            'keep' : keep, 'wins': wins,
            'sel_alpha': sel.alpha_,
            'sel_l1'   : sel.l1_ratio_,
            'final_alpha': halving.best_params_['enet__alpha'],
            'final_l1'   : halving.best_params_['enet__l1_ratio'],
            'feats2' : feats2
          }

    # re‑fit and test on held‑out city …


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model      import ElasticNetCV, HistGradientBoostingRegressor
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler
from sklearn.model_selection   import GroupKFold, cross_val_score
from sklearn.metrics           import r2_score

# ── Assumes in scope:
#    data_z, corrs, filter_features,
#    REDUNDANCY_R, SAME_SIGN_FRAC

# 1) Hyper‐parameter grids
keep_grid       = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid       = [6, 7, 8, 9, 10]
hgb_lr_grid     = [0.01, 0.1, 0.2]
hgb_depth_grid  = [3, 5, 7]
hgb_iter_grid   = [100, 200, 500]
# grids for the ElasticNetCV *filter*
sel_alpha_grid  = [0.001, 0.01, 0.1, 1.0]
sel_l1_grid     = [0.2, 0.5, 0.8, 1.0]

# 2) Prepare LOCO splits
groups    = data_z['city'].values
cities    = np.unique(groups)
outer_cv  = GroupKFold(n_splits=len(cities))
y_all     = data_z['trips_per_day'].values

outer_scores = []

# 3) Outer loop: leave one city out
for train_idx, test_idx in outer_cv.split(data_z, groups=groups):
    train    = data_z.iloc[train_idx]
    test     = data_z.iloc[test_idx]
    y_train  = train['trips_per_day'].values
    y_test   = test ['trips_per_day'].values

    # inner LOCO on the *training* cities
    inner_groups = train['city'].values
    inner_cv     = GroupKFold(n_splits=len(np.unique(inner_groups)))

    hgb_best_inner_r2 = -np.inf
    hgb_best_cfg      = None

    # 4) Inner loop: tune keep/wins + ENetCV filter + HGB hyperparams
    for keep in keep_grid:
        for wins in wins_grid:
            # 4a) corr→collinear→same‐sign
            feats1 = filter_features(
                corrs, train,
                keep_thresh     = keep,
                wins_thresh     = wins,
                colinear_thresh = REDUNDANCY_R,
                same_sign_frac  = SAME_SIGN_FRAC
            )
            if len(feats1) < 5:
                continue

            X1 = train[feats1].values

            # 4b) ElasticNetCV filter on FEATURES1
            enet_cv = Pipeline([
                ('scale', StandardScaler()),
                ('sel',   ElasticNetCV(
                              alphas       = sel_alpha_grid,
                              l1_ratio     = sel_l1_grid,
                              cv           = 5,
                              random_state = 42,
                              n_jobs       = -1
                          ))
            ]).fit(X1, y_train).named_steps['sel']

            coefs  = pd.Series(enet_cv.coef_, index=feats1)
            feats2 = coefs[coefs != 0].index.tolist()
            if len(feats2) < 5:
                continue

            X2 = train[feats2].values

            # 4c) HGB grid on FEATURES2
            for lr in hgb_lr_grid:
                for depth in hgb_depth_grid:
                    for iters in hgb_iter_grid:
                        hgb = HistGradientBoostingRegressor(
                            learning_rate = lr,
                            max_depth     = depth,
                            max_iter      = iters,
                            random_state  = 42
                        )
                        pipe = Pipeline([
                            ('scale', StandardScaler()),
                            ('hgb',   hgb)
                        ])
                        inner_scores = cross_val_score(
                            pipe, X2, y_train,
                            cv=inner_cv.split(train, groups=inner_groups),
                            scoring='r2', n_jobs=-1
                        )
                        mean_inner = inner_scores.mean()

                        if mean_inner > hgb_best_inner_r2:
                            hgb_best_inner_r2 = mean_inner
                            hgb_best_cfg = {
                                'keep'        : keep,
                                'wins'        : wins,
                                'sel_alpha'   : enet_cv.alpha_,
                                'sel_l1'      : enet_cv.l1_ratio_,
                                'lr'      : lr,
                                'depth'   : depth,
                                'iter'    : iters,
                                'feats2'      : feats2
                            }

    # end inner loops
    if hgb_best_cfg is None:
        raise RuntimeError("No valid configuration found in inner tuning")

    # 5) Re‑fit filter + final HGB on full TRAIN using best inner config
    feats2       = hgb_best_cfg['feats2']
    X_train_best = train[feats2].values
    final_pipe   = Pipeline([
        ('scale', StandardScaler()),
        ('hgb',   HistGradientBoostingRegressor(
                      learning_rate = hgb_best_cfg['lr'],
                      max_depth     = hgb_best_cfg['depth'],
                      max_iter      = hgb_best_cfg['iter'],
                      random_state  = 42
                  ))
    ])
    final_pipe.fit(X_train_best, y_train)

    # 6) Evaluate on held‑out CITY
    X_test_best = test[feats2].values
    y_pred      = final_pipe.predict(X_test_best)
    outer_scores.append(r2_score(y_test, y_pred))

# 7) Summarize true nested LOCO R²
outer_scores = np.array(outer_scores)
print(f"Nested LOCO R² = {outer_scores.mean():.4f} ± {outer_scores.std(ddof=1):.4f}")

# 8) Optionally inspect the best configuration found in the first fold
print("Example best‑inner config:", hgb_best_cfg)


In [ ]:
# Peeky HGB tuning with enet filter
import numpy as np
import pandas as pd
from sklearn.model_selection       import GroupKFold, cross_val_score
from sklearn.pipeline              import Pipeline
from sklearn.preprocessing         import StandardScaler
from sklearn.linear_model          import ElasticNetCV
from sklearn.ensemble              import HistGradientBoostingRegressor

# ── Assumes:
#    data_z, corrs, filter_features,
#    REDUNDANCY_R, SAME_SIGN_FRAC
groups      = data_z['city'].values
outer_cv    = GroupKFold(n_splits=len(np.unique(groups)))
y           = data_z['trips_per_day'].values

keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [6, 7, 8, 9, 10]

# HGB parameter grid
hgb_lr_grid    = [0.01, 0.1, 0.2]
hgb_depth_grid = [3, 5, 7]
hgb_iter_grid  = [100, 200, 500]

# also reuse these for the ENetCV filter
alpha_grid = [0.001, 0.01, 0.1, 1.0]
l1_grid    = [0.2, 0.5, 0.8, 1.0]

hgb_results     = []
hgb_best_score  = -np.inf
hgb_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")

        # 1–3) initial corr/collinearity/same-sign filter
        feats1 = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feats1))
        if len(feats1) < 5:
            print("  → too few features, skipping.")
            continue

        # 4) ElasticNetCV filter on those survivors
        X_sel = data_z[feats1].values
        enet_sel = Pipeline([
            ('scale',    StandardScaler()),
            ('enet_sel', ElasticNetCV(
                              l1_ratio     = l1_grid,
                              alphas       = alpha_grid,
                              cv           = 5,
                              random_state = 42,
                              n_jobs       = -1
                          ))
        ]).fit(X_sel, y)

        coefs    = pd.Series(enet_sel.named_steps['enet_sel'].coef_, index=feats1)
        feats2   = coefs[coefs != 0].index.tolist()
        print(" after ElasticNetCV filter:", len(feats2))
        if len(feats2) < 5:
            print("  → too few features after ENetCV, skipping.")
            continue

        # 5) assemble final matrix (raw + any precomputed lags)
        X_hgb_model = data_z[feats2].values

        # 6) now tune HGB hyperparameters
        for lr in hgb_lr_grid:
            for depth in hgb_depth_grid:
                for iters in hgb_iter_grid:
                    hgb = HistGradientBoostingRegressor(
                              learning_rate=lr,
                              max_depth    =depth,
                              max_iter     =iters,
                              random_state =42
                          )
                    pipe = Pipeline([
                        ('scale', StandardScaler()),
                        ('hgb',   hgb)
                    ])
                    scores = cross_val_score(
                        pipe, X_hgb_model, y,
                        cv=outer_cv.split(data_z, groups=groups),
                        scoring='r2', n_jobs=-1
                    )
                    mean_r2 = scores.mean()
                    print(f"   lr={lr:.2f}, depth={depth}, iters={iters} → R²={mean_r2:.4f}")

                    hgb_results.append({
                        'keep'         : keep,
                        'wins'         : wins,
                        'n_prefilt'    : len(feats1),
                        'n_enetsel'    : len(feats2),
                        'learning_rate': lr,
                        'max_depth'    : depth,
                        'max_iter'     : iters,
                        'r2'           : mean_r2
                    })
                    if mean_r2 > hgb_best_score:
                        hgb_best_score  = mean_r2
                        hgb_best_config = hgb_results[-1]

# final summary
print("\n=== Best HGB config ===")
print(hgb_best_config)

hgb_df = pd.DataFrame(hgb_results)
print("\nTop 5 HGB results by R²:")
print(hgb_df.sort_values('r2', ascending=False).head(5))


In [ ]:
# HGB Everything tuning no filter
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.ensemble import HistGradientBoostingRegressor

# ── Assumes:
#     data_z       : z-scored DataFrame with centroid_x, centroid_y, 'trips_per_day'
#     corrs        : feature×city correlation DataFrame
#     filter_features(...) from earlier cell
#     REDUNDANCY_R, SAME_SIGN_FRAC defined

# 0) setup
groups       = data_z['city'].values
outer_cv     = GroupKFold(n_splits=len(np.unique(groups)))
y            = data_z['trips_per_day']

# coords       = data_z[['centroid_x', 'centroid_y']].values
# nbrs_model   = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
# _, idxs      = nbrs_model.kneighbors(coords)

keep_grid    = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid    = [6, 7, 8, 9, 10]

# HGB parameter grid
hgb_lr_grid     = [0.01, 0.1, 0.2]
hgb_depth_grid  = [3, 5, 7]
hgb_iter_grid   = [100, 200, 500]

hgb_results      = []
hgb_best_score   = -np.inf
hgb_best_config  = None

for hgb_keep in keep_grid:
    for hgb_wins in wins_grid:
        print(f"\n=== hgb_keep={hgb_keep:.2f}, hgb_wins={hgb_wins} ===")
        
        # 1–3) correlation + collinearity + same-sign filtering
        hgb_feat1 = filter_features(
            corrs, data_z,
            keep_thresh=hgb_keep,
            wins_thresh=hgb_wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(hgb_feat1))
        if len(hgb_feat1) < 5:
            print("  → too few features after initial filter, skipping.")
            continue

        # 4) LassoCV pruning
        X_hgb_lasso = data_z[hgb_feat1].values
        if X_hgb_lasso.shape[1] == 0:
            print("  → no features to feed LassoCV, skipping.")
            continue

        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_hgb_lasso, y)
        hgb_feat2 = [f for f, coef in zip(hgb_feat1, lasso.coef_) if coef != 0]
        print(" after LassoCV:", len(hgb_feat2))

        X_hgb_model = data_z[hgb_feat2].values
        # if len(hgb_feat2) < 5:
        #     print("  → fewer than 5 Lasso-selected features, skipping.")
        #     continue

        # # 5) build 5-NN lag features
        # hgb_base_arrays = [data_z[col].values for col in hgb_feat2]
        # hgb_lag_arrays  = [
        #     np.array([arr[neighbors[1:]].mean() for neighbors in idxs])
        #     for arr in hgb_base_arrays
        # ]
        # X_hgb_base  = np.column_stack(hgb_base_arrays)
        # X_hgb_lag   = np.column_stack(hgb_lag_arrays)
        # X_hgb_model = np.hstack([X_hgb_base, X_hgb_lag])

        if X_hgb_model.shape[1] < 2:
            print("  → too few total features for HGB, skipping.")
            continue

        # 6) tune HGB parameters
        for hgb_lr in hgb_lr_grid:
            for hgb_depth in hgb_depth_grid:
                for hgb_iter in hgb_iter_grid:
                    hgb_model = HistGradientBoostingRegressor(
                        learning_rate=hgb_lr,
                        max_depth=hgb_depth,
                        max_iter=hgb_iter,
                        random_state=42
                    )
                    pipe = Pipeline([
                        ('scaler', StandardScaler()),  
                        ('hgb',    hgb_model)
                    ])
                    scores = cross_val_score(
                        pipe, X_hgb_model, y,
                        cv=outer_cv.split(data_z, groups=groups),
                        scoring='r2', n_jobs=-1
                    )
                    hgb_mean_r2 = scores.mean()
                    print(f"  lr={hgb_lr:.2f}, depth={hgb_depth}, iters={hgb_iter} → R²={hgb_mean_r2:.4f}")

                    hgb_results.append({
                        'hgb_keep': hgb_keep,
                        'hgb_wins': hgb_wins,
                        'learning_rate': hgb_lr,
                        'max_depth': hgb_depth,
                        'max_iter': hgb_iter,
                        'r2': hgb_mean_r2,
                        'corr_feats': len(hgb_feat1),
                        'lasso_feats': len(hgb_feat2),
                    })
                    if hgb_mean_r2 > hgb_best_score:
                        hgb_best_score  = hgb_mean_r2
                        hgb_best_config = hgb_results[-1]

# final summary
print("\n=== hgb Best config ===")
print(hgb_best_config)

hgb_df = pd.DataFrame(hgb_results)
print(hgb_df.sort_values('r2', ascending=False).head())


In [ ]:
# HGB everything tuning with lag building
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors

# ── Assumes:
#     data_z       : z-scored DataFrame with centroid_x, centroid_y, 'trips_per_day'
#     corrs        : feature×city correlation DataFrame
#     filter_features(...) from earlier cell
#     REDUNDANCY_R, SAME_SIGN_FRAC defined

# 0) setup
groups       = data_z['city'].values
outer_cv     = GroupKFold(n_splits=len(np.unique(groups)))
y            = data_z['trips_per_day']

coords       = data_z[['centroid_x', 'centroid_y']].values
nbrs_model   = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs      = nbrs_model.kneighbors(coords)

keep_grid    = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid    = [6, 7, 8, 9, 10]

# HGB parameter grid
hgb_lr_grid     = [0.01, 0.1, 0.2]
hgb_depth_grid  = [3, 5, 7]
hgb_iter_grid   = [100, 200, 500]

hgb_results      = []
hgb_best_score   = -np.inf
hgb_best_config  = None

for hgb_keep in keep_grid:
    for hgb_wins in wins_grid:
        print(f"\n=== hgb_keep={hgb_keep:.2f}, hgb_wins={hgb_wins} ===")
        
        # 1–3) correlation + collinearity + same-sign filtering
        hgb_feat1 = filter_features(
            corrs, data_z,
            keep_thresh=hgb_keep,
            wins_thresh=hgb_wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(hgb_feat1))
        if len(hgb_feat1) < 5:
            print("  → too few features after initial filter, skipping.")
            continue

        # 4) LassoCV pruning
        X_hgb_lasso = data_z[hgb_feat1].values
        if X_hgb_lasso.shape[1] == 0:
            print("  → no features to feed LassoCV, skipping.")
            continue

        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_hgb_lasso, y)
        hgb_feat2 = [f for f, coef in zip(hgb_feat1, lasso.coef_) if coef != 0]
        print(" after LassoCV:", len(hgb_feat2))
        if len(hgb_feat2) < 5:
            print("  → fewer than 5 Lasso-selected features, skipping.")
            continue

        # 5) build 5-NN lag features
        hgb_base_arrays = [data_z[col].values for col in hgb_feat2]
        hgb_lag_arrays  = [
            np.array([arr[neighbors[1:]].mean() for neighbors in idxs])
            for arr in hgb_base_arrays
        ]
        X_hgb_base  = np.column_stack(hgb_base_arrays)
        X_hgb_lag   = np.column_stack(hgb_lag_arrays)
        X_hgb_model = np.hstack([X_hgb_base, X_hgb_lag])

        if X_hgb_model.shape[1] < 2:
            print("  → too few total features for HGB, skipping.")
            continue

        # 6) tune HGB parameters
        for hgb_lr in hgb_lr_grid:
            for hgb_depth in hgb_depth_grid:
                for hgb_iter in hgb_iter_grid:
                    hgb_model = HistGradientBoostingRegressor(
                        learning_rate=hgb_lr,
                        max_depth=hgb_depth,
                        max_iter=hgb_iter,
                        random_state=42
                    )
                    pipe = Pipeline([
                        ('scaler', StandardScaler()),  
                        ('hgb',    hgb_model)
                    ])
                    scores = cross_val_score(
                        pipe, X_hgb_model, y,
                        cv=outer_cv.split(data_z, groups=groups),
                        scoring='r2', n_jobs=-1
                    )
                    hgb_mean_r2 = scores.mean()
                    print(f"  lr={hgb_lr:.2f}, depth={hgb_depth}, iters={hgb_iter} → R²={hgb_mean_r2:.4f}")

                    hgb_results.append({
                        'hgb_keep': hgb_keep,
                        'hgb_wins': hgb_wins,
                        'learning_rate': hgb_lr,
                        'max_depth': hgb_depth,
                        'max_iter': hgb_iter,
                        'r2': hgb_mean_r2,
                        'corr_feats': len(hgb_feat1),
                        'lasso_feats': len(hgb_feat2),
                    })
                    if hgb_mean_r2 > hgb_best_score:
                        hgb_best_score  = hgb_mean_r2
                        hgb_best_config = hgb_results[-1]

# final summary
print("\n=== hgb Best config ===")
print(hgb_best_config)

hgb_df = pd.DataFrame(hgb_results)
print(hgb_df.sort_values('r2', ascending=False).head())


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model       import ElasticNetCV, Ridge
from sklearn.pipeline           import Pipeline
from sklearn.preprocessing      import StandardScaler
from sklearn.model_selection    import GroupKFold, cross_val_score

# ── Assumes in scope:
#     data_z, corrs, filter_features,
#     REDUNDANCY_R, SAME_SIGN_FRAC

# 1) grids
keep_grid       = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid       = [6, 7, 8, 9, 10]
# Ridge α grid
ridge_alpha_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
# ENetCV filter grids
sel_alpha_grid  = [0.001, 0.01, 0.1, 1.0]
sel_l1_grid     = [0.2, 0.5, 0.8, 1.0]

# 2) LOCO CV splitter
groups    = data_z['city'].values
outer_cv  = GroupKFold(n_splits=len(np.unique(groups)))
y         = data_z['trips_per_day'].values

# 3) storage for results
ridge_results        = []
ridge_best_inner_r2  = -np.inf
ridge_config         = None

# 4) loop over corr‐thresh / wins‐thresh
for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")

        # 4a) corr→collinear→same‐sign
        feats1 = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feats1))
        if len(feats1) < 5:
            print("  → too few features, skipping")
            continue

        # 4b) ENetCV filter on feats1
        X_sel = data_z[feats1].values
        enet_sel = Pipeline([
            ('scale', StandardScaler()),
            ('enet_sel', ElasticNetCV(
                              alphas       = sel_alpha_grid,
                              l1_ratio     = sel_l1_grid,
                              cv           = 5,
                              random_state = 42,
                              n_jobs       = -1
                          ))
        ]).fit(X_sel, y).named_steps['enet_sel']

        coefs1 = pd.Series(enet_sel.coef_, index=feats1)
        feats2 = coefs1[coefs1 != 0].index.tolist()
        print(" after ElasticNetCV filter:", len(feats2))
        if len(feats2) < 5:
            print("  → too few features after ENetCV, skipping")
            continue

        # 4c) now tune Ridge on feats2
        X_ridge = data_z[feats2].values
        for alpha in ridge_alpha_grid:
            pipe = Pipeline([
                ('scale', StandardScaler()),
                ('ridge', Ridge(alpha=alpha))
            ])
            scores = cross_val_score(
                pipe, X_ridge, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2', n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"   α={alpha:.3f} → R²={mean_r2:.4f}")

            ridge_results.append({
                'keep'       : keep,
                'wins'       : wins,
                'n_prefilt'  : len(feats1),
                'n_enetsel'  : len(feats2),
                'sel_alpha'  : enet_sel.alpha_,
                'sel_l1'     : enet_sel.l1_ratio_,
                'alpha': alpha,
                'r2'         : mean_r2
            })
            if mean_r2 > ridge_best_inner_r2:
                ridge_best_inner_r2 = mean_r2
                ridge_config = ridge_results[-1]

# 5) summary
print("\n=== Best Ridge config ===")
print("ridge_best_inner_r2 =", ridge_best_inner_r2)
print("ridge_config =", ridge_config)

# optional: inspect top results
ridge_df = pd.DataFrame(ridge_results)
print("\nTop 5 by R²:")
print(ridge_df.sort_values('r2', ascending=False).head(5))


In [ ]:
# RidgeCV with enet tuning
import numpy as np
import pandas as pd
from sklearn.model_selection       import GroupKFold, cross_val_score
from sklearn.pipeline              import Pipeline
from sklearn.preprocessing         import StandardScaler
from sklearn.linear_model          import Ridge, ElasticNetCV
from sklearn.neighbors             import NearestNeighbors

# ── Assumes in scope:
#    data_z, corrs, filter_features,
#    REDUNDANCY_R, SAME_SIGN_FRAC
groups       = data_z['city'].values
outer_cv     = GroupKFold(n_splits=len(np.unique(groups)))
y            = data_z['trips_per_day'].values

keep_grid    = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid    = [5, 6, 7, 8, 9, 10]
alpha_grid   = [0.01, 0.1, 1.0, 10.0, 100.0]   # Ridge α’s

# grids reused for the ElasticNetCV filter
enet_alpha_grid = [0.001, 0.01, 0.1, 1.0]
enet_l1_grid    = [0.2, 0.5, 0.8, 1.0]

ridge_results     = []
ridge_best_score  = -np.inf
ridge_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")
        
        # 1–3) corr + collinear + same‑sign
        feat1 = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feat1))
        if len(feat1) < 5:
            print("  → too few features, skipping.")
            continue

        # 4) ElasticNetCV filter on feat1
        X_prefilt = data_z[feat1].values
        enet_sel = Pipeline([
            ('scale',    StandardScaler()),
            ('enet_sel', ElasticNetCV(
                              l1_ratio     = enet_l1_grid,
                              alphas       = enet_alpha_grid,
                              cv           = 5,
                              random_state = 42,
                              n_jobs       = -1
                          ))
        ]).fit(X_prefilt, y)

        coefs  = pd.Series(
                    enet_sel.named_steps['enet_sel'].coef_,
                    index=feat1
                 )
        feat2  = coefs[coefs != 0].index.tolist()
        print(" after ElasticNetCV filter:", len(feat2))
        if len(feat2) < 5:
            print("  → too few after ENetCV, skipping.")
            continue

        # 5) build final matrix for Ridge
        X_model = data_z[feat2].values

        # 6) tune Ridge alpha over your grid
        for alpha in alpha_grid:
            pipe = Pipeline([
                ('scale', StandardScaler()),
                ('ridge', Ridge(alpha=alpha))
            ])
            scores = cross_val_score(
                pipe, X_model, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2',
                n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"  Ridge α={alpha:.3f} → R²={mean_r2:.4f}")

            ridge_results.append({
                'keep'         : keep,
                'wins'         : wins,
                'n_prefilter'  : len(feat1),
                'n_enetsel'    : len(feat2),
                'alpha'        : alpha,
                'r2'           : mean_r2
            })
            if mean_r2 > ridge_best_score:
                ridge_best_score  = mean_r2
                ridge_best_config = ridge_results[-1]

# final summary
print("\n=== Ridge Best config ===")
print(ridge_best_config)

ridge_df = pd.DataFrame(ridge_results)
print("\nTop 5 Ridge results by R²:")
print(ridge_df.sort_values('r2', ascending=False).head(5))


In [ ]:
# RidgeCV everything tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.linear_model import LassoCV
from sklearn.neighbors import NearestNeighbors

# ── Assumes:
#     data_z       : z-scored DataFrame with centroid_x, centroid_y, 'trips_per_day'
#     corrs        : feature×city correlation DataFrame
#     filter_features(...) from earlier cell
#     REDUNDANCY_R, SAME_SIGN_FRAC defined

# 0) setup
groups      = data_z['city'].values
outer_cv    = GroupKFold(n_splits=len(np.unique(groups)))
y           = data_z['trips_per_day']

# coords      = data_z[['centroid_x', 'centroid_y']].values
# nbrs_model  = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
# _, idxs     = nbrs_model.kneighbors(coords)

keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [5, 6, 7, 8, 9, 10]
alpha_grid  = [0.01, 0.1, 1.0, 10.0, 100.0]   # Ridge regularization strengths

ridge_results     = []
ridge_best_score  = -np.inf
ridge_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")
        
        # 1–3) correlation + collinearity + same-sign filtering
        feat1 = filter_features(
            corrs, data_z,
            keep_thresh=keep,
            wins_thresh=wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feat1))
        if len(feat1) < 5:
            continue
        
        X_model = data_z[feat1]
        # 4) LassoCV pruning
        # X_lasso = data_z[feat1].values
        # lasso   = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        # feat2   = [f for f, coef in zip(feat1, lasso.coef_) if coef != 0]
        # print(" after LassoCV:", len(feat2))

        # X_model = data_z[feat2]
        # # if len(feat2) < 5:
        # #     continue

        # # 5) build 5-NN lag features
        # X_base = data_z[feat2].values
        # X_lag  = np.column_stack([
        #     np.array([data_z[col].values[nbrs[1:]].mean()
        #               for nbrs in idxs])
        #     for col in feat2
        # ])
        # X_model = np.hstack([X_base, X_lag])
        if X_model.shape[1] < 2:
            continue

        # 6) tune Ridge alpha
        for alpha in alpha_grid:
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('ridge',  Ridge(alpha=alpha))
            ])
            scores = cross_val_score(
                pipe, X_model, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2', n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"  alpha={alpha:.3f} → R²={mean_r2:.4f}")

            ridge_results.append({
                'keep': keep,
                'wins': wins,
                'alpha': alpha,
                'r2': mean_r2,
                'corr_feats': len(feat1),
#                'lasso_feats': len(feat2),
            })
            if mean_r2 > ridge_best_score:
                ridge_best_score  = mean_r2
                ridge_best_config = ridge_results[-1]

# final summary
print("\n=== Ridge Best config ===")
print(ridge_best_config)

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.sort_values('r2', ascending=False).head())


In [ ]:
# RidgeCV everything tuning with lag building
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.linear_model import LassoCV
from sklearn.neighbors import NearestNeighbors

# ── Assumes:
#     data_z       : z-scored DataFrame with centroid_x, centroid_y, 'trips_per_day'
#     corrs        : feature×city correlation DataFrame
#     filter_features(...) from earlier cell
#     REDUNDANCY_R, SAME_SIGN_FRAC defined

# 0) setup
groups      = data_z['city'].values
outer_cv    = GroupKFold(n_splits=len(np.unique(groups)))
y           = data_z['trips_per_day']

coords      = data_z[['centroid_x', 'centroid_y']].values
nbrs_model  = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs     = nbrs_model.kneighbors(coords)

keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [5, 6, 7, 8, 9, 10]
alpha_grid  = [0.01, 0.1, 1.0, 10.0, 100.0]   # Ridge regularization strengths

ridge_results     = []
ridge_best_score  = -np.inf
ridge_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")
        
        # 1–3) correlation + collinearity + same-sign filtering
        feat1 = filter_features(
            corrs, data_z,
            keep_thresh=keep,
            wins_thresh=wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feat1))
        if len(feat1) < 5:
            continue

        # 4) LassoCV pruning
        X_lasso = data_z[feat1].values
        lasso   = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        feat2   = [f for f, coef in zip(feat1, lasso.coef_) if coef != 0]
        print(" after LassoCV:", len(feat2))
        if len(feat2) < 5:
            continue

        # 5) build 5-NN lag features
        X_base = data_z[feat2].values
        X_lag  = np.column_stack([
            np.array([data_z[col].values[nbrs[1:]].mean()
                      for nbrs in idxs])
            for col in feat2
        ])
        X_model = np.hstack([X_base, X_lag])
        if X_model.shape[1] < 2:
            continue

        # 6) tune Ridge alpha
        for alpha in alpha_grid:
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('ridge',  Ridge(alpha=alpha))
            ])
            scores = cross_val_score(
                pipe, X_model, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2', n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"  alpha={alpha:.3f} → R²={mean_r2:.4f}")

            ridge_results.append({
                'keep': keep,
                'wins': wins,
                'alpha': alpha,
                'r2': mean_r2,
                'corr_feats': len(feat1),
                'lasso_feats': len(feat2),
            })
            if mean_r2 > ridge_best_score:
                ridge_best_score  = mean_r2
                ridge_best_config = ridge_results[-1]

# final summary
print("\n=== Ridge Best config ===")
print(ridge_best_config)

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.sort_values('r2', ascending=False).head())


In [ ]:
# PLS everything tuning with enet filter 
import numpy as np
import pandas as pd
from sklearn.model_selection       import GroupKFold, cross_val_score
from sklearn.pipeline              import Pipeline
from sklearn.preprocessing         import StandardScaler
from sklearn.linear_model          import ElasticNetCV
from sklearn.cross_decomposition   import PLSRegression

# ── Assumes in scope:
#    data_z, corrs, filter_features,
#    REDUNDANCY_R, SAME_SIGN_FRAC

groups         = data_z['city'].values
outer_cv       = GroupKFold(n_splits=len(np.unique(groups)))
y              = data_z['trips_per_day'].values

keep_grid      = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid      = [5, 6, 7, 8, 9, 10]
ncomp_grid     = [2, 5, 10, 15, 20]      # number of PLS components

# reuse these for the ElasticNetCV filter
enet_alpha_grid = [0.001, 0.01, 0.1, 1.0]
enet_l1_grid    = [0.2, 0.5, 0.8, 1.0]

pls_results     = []
pls_best_score  = -np.inf
pls_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")

        # 1–3) corr + collinear + same-sign
        feat1 = filter_features(
            corrs, data_z,
            keep_thresh     = keep,
            wins_thresh     = wins,
            colinear_thresh = REDUNDANCY_R,
            same_sign_frac  = SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feat1))
        if len(feat1) < 5:
            print("  → skipping, too few features")
            continue

        # 4) ElasticNetCV filter on feat1
        X_prefilt = data_z[feat1].values
        enet_sel = Pipeline([
            ('scale',    StandardScaler()),
            ('enet_sel', ElasticNetCV(
                              l1_ratio     = enet_l1_grid,
                              alphas       = enet_alpha_grid,
                              cv           = 5,
                              random_state = 42,
                              n_jobs       = -1
                          ))
        ]).fit(X_prefilt, y)

        coefs  = pd.Series(
            enet_sel.named_steps['enet_sel'].coef_,
            index=feat1
        )
        feat2  = coefs[coefs != 0].index.tolist()
        print(" after ElasticNetCV filter:", len(feat2))
        if len(feat2) < 5:
            print("  → skipping, too few after ENetCV")
            continue

        # 5) assemble final matrix for PLS
        X_model = data_z[feat2].values

        # 6) tune PLS n_components
        for n_comp in ncomp_grid:
            if n_comp >= X_model.shape[1]:
                continue
            pls = PLSRegression(n_components=n_comp)
            pipe = Pipeline([
                ('scale', StandardScaler()),
                ('pls',   pls)
            ])
            scores = cross_val_score(
                pipe, X_model, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2',
                n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"   n_components={n_comp} → R²={mean_r2:.4f}")

            pls_results.append({
                'keep'         : keep,
                'wins'         : wins,
                'n_prefilter'  : len(feat1),
                'n_enetsel'    : len(feat2),
                'n_components': n_comp,
                'r2'          : mean_r2
            })
            if mean_r2 > pls_best_score:
                pls_best_score  = mean_r2
                pls_best_config = pls_results[-1]

# final summary
print("\n=== PLSRegression Best config ===")
print(pls_best_config)

pls_df = pd.DataFrame(pls_results)
print("\nTop 5 PLS results by R²:")
print(pls_df.sort_values('r2', ascending=False).head(5))

In [ ]:
# PLS everything tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors

# ── Assumes:
#     data_z       : z-scored DataFrame with centroid_x, centroid_y, 'trips_per_day'
#     corrs        : feature×city correlation DataFrame
#     filter_features(...) from earlier cell
#     REDUNDANCY_R, SAME_SIGN_FRAC defined

# 0) setup
groups      = data_z['city'].values
outer_cv    = GroupKFold(n_splits=len(np.unique(groups)))
y           = data_z['trips_per_day']


# coords      = data_z[['centroid_x', 'centroid_y']].values
# nbrs_model  = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
# _, idxs     = nbrs_model.kneighbors(coords)

keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [5, 6, 7, 8, 9, 10]
ncomp_grid  = [2, 5, 10, 15, 20]  # number of PLS components

pls_results     = []
pls_best_score  = -np.inf
pls_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")
        
        # 1–3) correlation + collinearity + same-sign filtering
        feat1 = filter_features(
            corrs, data_z,
            keep_thresh=keep,
            wins_thresh=wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feat1))
        if len(feat1) < 5:
            continue

        # 4) LassoCV pruning
        X_lasso = data_z[feat1].values
        lasso   = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        feat2   = [f for f, coef in zip(feat1, lasso.coef_) if coef != 0]
        print(" after LassoCV:", len(feat2))
        X_model = data_z[feat2].values
        # if len(feat2) < 5:
        #     continue

        # # 5) build 5-NN lag features
        # X_base = data_z[feat2].values
        # X_lag  = np.column_stack([
        #     np.array([data_z[col].values[nbrs[1:]].mean()
        #               for nbrs in idxs])
        #     for col in feat2
        # ])
        # X_model = np.hstack([X_base, X_lag])
        if X_model.shape[1] < 2:
            continue

        # 6) tune PLSRegression n_components
        for n_comp in ncomp_grid:
            # skip if n_components exceeds #features
            if n_comp >= X_model.shape[1]:
                continue

            pls = PLSRegression(n_components=n_comp)
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('pls',    pls)
            ])
            scores = cross_val_score(
                pipe, X_model, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2', n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"  n_components={n_comp} → R²={mean_r2:.4f}")

            pls_results.append({
                'keep': keep,
                'wins': wins,
                'n_components': n_comp,
                'r2': mean_r2,
                'corr_feats': len(feat1),
                'lasso_feats': len(feat2),
            })
            if mean_r2 > pls_best_score:
                pls_best_score  = mean_r2
                pls_best_config = pls_results[-1]

# final summary
print("\n=== PLSRegression Best config ===")
print(pls_best_config)

pls_df = pd.DataFrame(pls_results)
print(pls_df.sort_values('r2', ascending=False).head())


In [ ]:
# PLS everything tuning with lag building
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors

# ── Assumes:
#     data_z       : z-scored DataFrame with centroid_x, centroid_y, 'trips_per_day'
#     corrs        : feature×city correlation DataFrame
#     filter_features(...) from earlier cell
#     REDUNDANCY_R, SAME_SIGN_FRAC defined

# 0) setup
groups      = data_z['city'].values
outer_cv    = GroupKFold(n_splits=len(np.unique(groups)))
#y           = np.log1p(data_z['trips_per_day'].values)
y           = data_z['trips_per_day']


coords      = data_z[['centroid_x', 'centroid_y']].values
nbrs_model  = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs     = nbrs_model.kneighbors(coords)

keep_grid   = [0.10, 0.15, 0.20, 0.25, 0.30]
wins_grid   = [5, 6, 7, 8, 9, 10]
ncomp_grid  = [2, 5, 10, 15, 20]  # number of PLS components

pls_results     = []
pls_best_score  = -np.inf
pls_best_config = None

for keep in keep_grid:
    for wins in wins_grid:
        print(f"\n=== keep={keep:.2f}, wins={wins} ===")
        
        # 1–3) correlation + collinearity + same-sign filtering
        feat1 = filter_features(
            corrs, data_z,
            keep_thresh=keep,
            wins_thresh=wins,
            colinear_thresh=REDUNDANCY_R,
            same_sign_frac=SAME_SIGN_FRAC
        )
        print(" after corr/collinear/same-sign:", len(feat1))
        if len(feat1) < 5:
            continue

        # 4) LassoCV pruning
        X_lasso = data_z[feat1].values
        lasso   = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        feat2   = [f for f, coef in zip(feat1, lasso.coef_) if coef != 0]
        print(" after LassoCV:", len(feat2))
        if len(feat2) < 5:
            continue

        # 5) build 5-NN lag features
        X_base = data_z[feat2].values
        X_lag  = np.column_stack([
            np.array([data_z[col].values[nbrs[1:]].mean()
                      for nbrs in idxs])
            for col in feat2
        ])
        X_model = np.hstack([X_base, X_lag])
        if X_model.shape[1] < 2:
            continue

        # 6) tune PLSRegression n_components
        for n_comp in ncomp_grid:
            # skip if n_components exceeds #features
            if n_comp >= X_model.shape[1]:
                continue

            pls = PLSRegression(n_components=n_comp)
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('pls',    pls)
            ])
            scores = cross_val_score(
                pipe, X_model, y,
                cv=outer_cv.split(data_z, groups=groups),
                scoring='r2', n_jobs=-1
            )
            mean_r2 = scores.mean()
            print(f"  n_components={n_comp} → R²={mean_r2:.4f}")

            pls_results.append({
                'keep': keep,
                'wins': wins,
                'n_components': n_comp,
                'r2': mean_r2,
                'corr_feats': len(feat1),
                'lasso_feats': len(feat2),
            })
            if mean_r2 > pls_best_score:
                pls_best_score  = mean_r2
                pls_best_config = pls_results[-1]

# final summary
print("\n=== PLSRegression Best config ===")
print(pls_best_config)

pls_df = pd.DataFrame(pls_results)
print(pls_df.sort_values('r2', ascending=False).head())


In [ ]:
import numpy as np
import pandas as pd
from sklearn.base            import clone
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import ElasticNetCV
from sklearn.model_selection import cross_val_score

def evaluate_with_thresholds(
    model,
    keep_thresh: float,
    wins_thresh: int,
    data_z,
    corrs,
    groups,
    outer_cv,
    y,
    colinear_thresh: float = REDUNDANCY_R,
    same_sign_frac: float = SAME_SIGN_FRAC,
    selector_alphas = [0.001, 0.01, 0.1, 1.0],
    selector_l1     = [0.2, 0.5, 0.8, 1.0]
):
    """
    Returns (best_mean_r2, best_feature_list) where best_mean_r2 is the
    highest LOCO R² achieved by any (alpha, l1_ratio) selector, and
    best_feature_list is the set of nonzero‐coef features for that selector.
    """
    # 1) initial corr→collinear→same-sign
    feat1 = filter_features(
        corrs, data_z,
        keep_thresh     = keep_thresh,
        wins_thresh     = wins_thresh,
        colinear_thresh = colinear_thresh,
        same_sign_frac  = same_sign_frac
    )
    if len(feat1) < 2:
        return np.nan, []

    X_prefilt = data_z[feat1].values

    enet_sel = Pipeline([
        ('scale',    StandardScaler()),
        ('enet_sel', ElasticNetCV(
        l1_ratio     = selector_l1,
        alphas       = selector_alphas,
        cv           = 5,
        random_state = 42,
        n_jobs       = -1
                        ))
    ]).fit(X_prefilt, y)

    coefs  = pd.Series(
        enet_sel.named_steps['enet_sel'].coef_,
        index=feat1
    )
    feat2  = coefs[coefs != 0].index.tolist()
    print(" after ElasticNetCV filter:", len(feat2))
    if len(feat2) < 5:
        print("  → skipping, too few after ENetCV")

    # 2b) evaluate downstream model on those features
    X_model = data_z[feat2].values
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('est',   clone(model))
    ])
    scores = cross_val_score(
        pipe, X_model, y,
        cv=outer_cv.split(data_z, groups=groups),
        scoring='r2', n_jobs=-1
    )
    mean_r2 = scores.mean()

    return mean_r2, feat2


In [ ]:
# import numpy as np
# from sklearn.pipeline            import Pipeline
# from sklearn.preprocessing       import StandardScaler
# from sklearn.linear_model        import ElasticNetCV
# from sklearn.model_selection     import cross_val_score

# def evaluate_with_thresholds(
#     model,
#     keep_thresh: float,
#     wins_thresh: int,
#     data_z,
#     corrs,
#     groups,
#     outer_cv,
#     y,
#     colinear_thresh: float = REDUNDANCY_R,
#     same_sign_frac: float = SAME_SIGN_FRAC,
#     # new grids for the ENetCV filter:
#     enet_alphas = [0.001, 0.01, 0.1, 1.0],
#     enet_l1     = [0.2, 0.5, 0.8, 1.0]
# ):
#     """
#     1) corr → collinearity → same-sign via filter_features()
#     2) ElasticNetCV filter
#     3) build 5-NN lag features
#     4) evaluate `model` under LOCO R²

#     Returns
#     -------
#     (mean_r2, feat_list) or (np.nan, [])
#     """
#     # 1) feature filtering
#     feat1 = filter_features(
#         corrs, data_z,
#         keep_thresh     = keep_thresh,
#         wins_thresh     = wins_thresh,
#         colinear_thresh = colinear_thresh,
#         same_sign_frac  = same_sign_frac
#     )
#     if len(feat1) < 5:
#         print(f"→ only {len(feat1)} after corr/col/same; skipping")
#         return np.nan, []

#     # 2) ElasticNetCV pruning
#     X1 = data_z[feat1].values
#     enet = ElasticNetCV(
#         l1_ratio    = enet_l1,
#         alphas      = enet_alphas,
#         cv          = 5,
#         random_state=42,
#         n_jobs      = -1
#     ).fit(X1, y)
#     coefs = pd.Series(enet.coef_, index=feat1)
#     feat2 = coefs[coefs != 0].index.tolist()
#     if len(feat2) < 5:
#         print(f"→ only {len(feat2)} after ENetCV; skipping")
#         return np.nan, []

#     X_model = data_z[feat2].values

#     # 4) LOCO R² evaluation
#     pipe = Pipeline([
#         ('scaler', StandardScaler()),
#         ('est',    model)
#     ])
#     scores = cross_val_score(
#         pipe, X_model, y,
#         cv=outer_cv.split(data_z, groups=groups),
#         scoring='r2',
#         n_jobs=-1
#     )
#     return scores.mean(), feat2


In [ ]:
# Lasso tuned model evaluation
# # Re-compute r2 for each model's best parameters and get feature list (no lags included)
# import numpy as np
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LassoCV
# from sklearn.model_selection import cross_val_score

# def evaluate_with_thresholds(
#     model,
#     keep_thresh: float,
#     wins_thresh: int,
#     data_z,
#     corrs,
#     groups,
#     idxs,
#     outer_cv,
#     y,
#     colinear_thresh: float = REDUNDANCY_R,
#     same_sign_frac: float = SAME_SIGN_FRAC
# ):
#     """
#     1) Applies the corr→collinear→same-sign filters via filter_features()
#     2) Prunes with LassoCV
#     3) Builds 5-NN lag features
#     4) Evaluates `model` (must be a regressor) under LOCO R²

#     Returns
#     -------
#     mean_r2 : float
#         average R² over outer_cv folds, or np.nan if insufficient features
#     """
#     # 1) feature filtering
#     feat1 = filter_features(
#         corrs, data_z,
#         keep_thresh=keep_thresh,
#         wins_thresh=wins_thresh,
#         colinear_thresh=colinear_thresh,
#         same_sign_frac=same_sign_frac
#     )
#     if len(feat1) < 5:
#         print(f"→ only {len(feat1)} features after filter; skipping")
#         return np.nan

#     # 2) LassoCV pruning
#     X_lasso = data_z[feat1].values
#     lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
#     feat2 = [f for f,coef in zip(feat1, lasso.coef_) if coef != 0]
#     if len(feat2) < 5:
#         print(f"→ only {len(feat2)} features after Lasso; skipping")
#         return np.nan

#     X_model = data_z[feat2]
#     # 3) build 5-NN lag features

#     # coords   = data_z[['centroid_x','centroid_y']].values
#     # nbrs_mod = NearestNeighbors(n_neighbors=6, algorithm='ball_tree') \
#     #         .fit(coords)
#     # _, idxs  = nbrs_mod.kneighbors(coords)

#     # X_base = data_z[feat2].values
#     # X_lag = np.column_stack([
#     #     np.array([data_z[col].values[nbrs[1:]].mean() for nbrs in idxs])
#     #     for col in feat2
#     # ])
#     #X_model = np.hstack([X_base, X_lag])
#     if X_model.shape[1] < 2:
#         print("→ too few total features after lags; skipping")
#         return np.nan

#     # 4) LOCO R² evaluation
#     pipe = Pipeline([
#         ('scaler', StandardScaler()),
#         ('est',    model)
#     ])
#     scores = cross_val_score(
#         pipe, X_model, y,
#         cv=outer_cv.split(data_z, groups=groups),
#         scoring='r2',
#         n_jobs=-1
#     )
#     return scores.mean(), feat2

In [ ]:
from sklearn.linear_model            import ElasticNet, RidgeCV
from sklearn.ensemble                import HistGradientBoostingRegressor, StackingRegressor
from sklearn.cross_decomposition     import PLSRegression
from collections                     import namedtuple

# 1) Define a simple container for each model’s config
ModelSpec = namedtuple('ModelSpec', [
    'name',           # for display & for stacking
    'estimator_cls',  # the sklearn class
    'estimator_kwargs',# dict of hyper‐params to instantiate
    'keep',           # best keep_thresh
    'wins'            # best wins_thresh
])

model_specs = [
    ModelSpec(
        name='enet',
        estimator_cls=ElasticNet,
        estimator_kwargs={
            'alpha': enet_best_config['alpha'],
            'l1_ratio': enet_best_config['l1_ratio'],
            'random_state':42,
            'max_iter':10_000
        },
        keep=enet_best_config['keep'],
        wins=enet_best_config['wins']
    ),

    ModelSpec(
        name='hgb',
        estimator_cls=HistGradientBoostingRegressor,
        estimator_kwargs={
            'learning_rate' : hgb_best_config['learning_rate'],
            'max_depth' : hgb_best_config['max_depth'],
            'max_iter' : hgb_best_config['max_iter'],
            'random_state':42
        },
        keep=hgb_best_config['keep'],
        wins=hgb_best_config['wins']

    ),
    ModelSpec(
        name='ridge',
        estimator_cls=RidgeCV,
        estimator_kwargs={'alphas': ridge_best_config['alpha']},
        keep=ridge_best_config['keep'],
        wins=ridge_best_config['wins']
    ),
    ModelSpec(
        name='pls',
        estimator_cls=PLSRegression,
        estimator_kwargs={'n_components' : pls_best_config['n_components']},
        keep=pls_best_config['keep'],
        wins=pls_best_config['wins']
    )
]

# 2) Loop once to evaluate each model and capture its selected features
results = {}
for spec in model_specs:
    # instantiate the estimator fresh
    est = spec.estimator_cls(**spec.estimator_kwargs)
    
    # runs filter→elasticnet→LOCO-R2 and returns (r2, [feat_names])
    mean_r2, feats = evaluate_with_thresholds(
        model       = est,
        keep_thresh = spec.keep,
        wins_thresh = spec.wins,
        data_z      = data_z,
        corrs       = corrs,
        groups      = data_z['city'].values,
        outer_cv    = outer_cv,
        y           = y,
        colinear_thresh = REDUNDANCY_R,
        same_sign_frac  = SAME_SIGN_FRAC
    )
    print(f"{spec.name:6s} → R² = {mean_r2:.3f}, {len(feats)} features")
    results[spec.name] = {
        'r2': mean_r2,
        'features': feats
    }

# 3) Build the union of all features (so we only compute lags once)
# union_feats = set().union(*(res['features'] for res in results.values()))

# # recompute X_full = [orig cols in union_feats] + their lags
# X_full = data_z[list(union_feats)].copy()

# enet best: {'keep': 0.1, 'wins': 11, 'alpha': 0.1, 'l1_ratio': 0.2, 'n_prefilt': 30, 'n_enetsel': 19, 'r2': 0.35336775933403786}

# hgb best: {'keep': 0.1, 'wins': 6, 'n_prefilt': 171, 'n_enetsel': 53, 'learning_rate': 0.2, 'max_depth': 5, 'max_iter': 100, 'r2': 0.4117356998605047}
# ridge best: {'keep': 0.1, 'wins': 5, 'n_prefilter': 176, 'n_enetsel': 54, 'alpha': 100.0, 'r2': 0.35010593791883404}
# {'keep': 0.1, 'wins': 5, 'n_prefilter': 176, 'n_enetsel': 54, 'n_components': 2, 'r2': 0.3851237341567293}



In [ ]:
# 3) Build the union of all features (so we only compute lags once)
union_feats = set().union(*(res['features'] for res in results.values()))

# recompute X_full = [orig cols in union_feats] + their lags
X_full = data_z[list(union_feats)].copy()

    
# 4) Now build pipelines for stacking, each selecting its own subset
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

estimators = []
for spec in model_specs:
    feat = results[spec.name]['features']
    selector = ColumnTransformer([
        ('pick', 'passthrough', feat)
    ], remainder='drop')
    
    pipe = Pipeline([
        ('select', selector),
        ('scale',  StandardScaler()),
        ('est',    spec.estimator_cls(**spec.estimator_kwargs))
    ])
    estimators.append((spec.name, pipe))

In [ ]:
# 5) Stack them
final_stack_ridge = StackingRegressor(
    estimators       = estimators,
    final_estimator  = RidgeCV(alphas=[0.01,0.1,1.0]),
    passthrough      = False,
    n_jobs           = -1
)

# 6) Evaluate ensemble LOCO‐R²
from sklearn.model_selection import cross_val_score

stack_score = cross_val_score(
    final_stack_ridge, X_full, y,
    cv=outer_cv.split(X_full, y, groups=data_z['city'].values),
    scoring='r2', n_jobs=-1
)

print(f"Ridge metalearner LOCO R² = {stack_score.mean():.4f} ± {stack_score.std(ddof=1):.4f}")

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LassoCV

final_stack_lasso = StackingRegressor(
    estimators      = estimators,               # your 4 pipelines
    final_estimator = LassoCV(
        cv=5,
        positive=True,                          # force non-neg weights
        random_state=42
    ),
    passthrough     = False,
    n_jobs          = -1
)

stack_score = cross_val_score(
    final_stack_lasso, X_full, y,
    cv=outer_cv.split(X_full, y, groups=data_z['city'].values),
    scoring='r2', n_jobs=-1
)
print(
    f"LassoCV metalearner LOCO R² = "
    f"{stack_score.mean():.4f} ± {stack_score.std(ddof=1):.4f}"
)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

final_stack_hgb = StackingRegressor(
    estimators      = estimators,
    final_estimator = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=3,
        max_iter=100,
        random_state=42
    ),
    passthrough     = False,
    n_jobs          = -1
)

stack_score = cross_val_score(
    final_stack_hgb, X_full, y,
    cv=outer_cv.split(X_full, y, groups=groups),
    scoring='r2', n_jobs=-1
)
print(
    f"HGB metalearner LOCO R² = "
    f"{stack_score.mean():.4f} ± {stack_score.std(ddof=1):.4f}"
)

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd

# 1) Make sure your final_stack is already fitted on all the data
final_stack_ridge.fit(X_full, y)

# 2) Compute permutation importances on the training set
perm = permutation_importance(
    final_stack_ridge, 
    X_full, 
    y, 
    n_repeats=10, 
    random_state=42, 
    n_jobs=-1,
    scoring='r2'
)

# 3) Turn it into a sorted Series
importances = pd.Series(
    perm.importances_mean, 
    index=X_full.columns
).sort_values(ascending=False)

# 4) Look at the top 20 features
print("Top 20 features by permutation importance:")
print(importances.head(20))



In [ ]:
importances.index

In [ ]:
from sklearn.model_selection   import cross_val_predict
from sklearn.metrics           import r2_score, mean_squared_error

# 1) Get LOCO predictions for all rows
y_oof = cross_val_predict(
    final_stack_ridge,
    X_full, y,
    groups=data_z['city'],
    cv=outer_cv,
    method='predict',
    n_jobs=-1
)

# 2) Build a DataFrame with actual vs predicted
df = data_z[['city']].copy()
df['y_true'] = y
df['y_pred'] = y_oof

# 3) Compute per-city LOCO R² and RMSE
city_scores = (
    df
      .groupby('city')
      .apply(lambda g: pd.Series({
          'r2':   r2_score(g['y_true'], g['y_pred']),
          'rmse': mean_squared_error(
                       g['y_true'], g['y_pred'], squared=False)
      }))
)

print(city_scores)

# 4) Compare overall LOCO R²
global_loco_r2 = r2_score(df['y_true'], df['y_pred'])
print("Global LOCO R² (all cities together):", global_loco_r2)


In [ ]:

# ------------------------------------------------------------------
# 1.  Load St Louis tracts + ACS exactly as in your train prep
# ------------------------------------------------------------------
from functools import reduce

# 1) Your STL tract geometry + minimal cols
mo_gdf = gpd.read_file("./tract_shapefiles/tl_2023_mo_tract/tl_2023_29_tract.shp")
stl_tr = mo_gdf.query("COUNTYFP in ['510','189']").copy()
stl_tr['geoid']      = stl_tr['GEOID'].str[-11:]
stl_tr['centroid_x'] = stl_tr.geometry.centroid.x
stl_tr['centroid_y'] = stl_tr.geometry.centroid.y
stl_tr = stl_tr.rename(columns={'ALAND':'aland'})[['geoid','aland','centroid_x','centroid_y']]

# 2) Load & keep *all* estimate columns (E), drop the M ones

census_files = [
    "./census_data/S0101_2023.csv",
    "./census_data/S0601_2023.csv",
    "./census_data/S0701_2023.csv",
    "./census_data/S0801_2023.csv",
    "./census_data/S0802_2020.csv",
    "./census_data/DP03_2023.csv",
]
frames = []
for path in census_files:
    df = pd.read_csv(path, dtype={"GEOID": str})
    df = df.rename(columns={"GEOID": "geoid", "GEO_ID": "geoid"})
    df["geoid"] = df["geoid"].str[-11:]
    # pick only columns that end with "E", *except* the tract name
    est_cols = [c for c in df.columns 
                if c.endswith("E") 
                and c != "NAME"         # <-- drop the NAME field
                and not c.endswith("M")] # also skip any margin‐of‐error
    frames.append(df[["geoid", *est_cols]])

# 3) Outer‐merge so we get one wide ACS frame
acs = reduce(lambda L, R: L.merge(R, on="geoid", how="outer"), frames)

# 4) Join onto your STL tracts
stl = stl_tr.merge(acs, on="geoid", how="left")

# quick check
print("Got population column?", "S0101_C01_001E" in stl.columns)
print("Nulls in population:", stl["S0101_C01_001E"].isna().sum())

In [ ]:
print("Original shape: " , stl.shape)
data = stl.dropna(how='all') #thresh = require that many non na values to not be dropped
print("After removing rows missing every entry: " , data.shape)
data = data.dropna(how='all', axis=1) #thresh = how many non na values to not be dropped
print("After removing cols missing every entry: " , data.shape)

exclude = ['geoid']
# all other columns: attempt to convert to numeric
for col in data.columns:
    if col not in exclude:
        data[col] = pd.to_numeric(data[col], errors='coerce')
print(data.shape)
data = data.dropna(axis=1, how='all')
print(data.shape)

for col in data.columns:
    if col == 'geoid':
        continue
    median = data[col].median()
    data.loc[:, col] = data[col].fillna(median)
print(data.shape)
data = data.dropna(axis=1, how='any')
print(data.shape)
total_nans = data.isna().sum().sum()
print(f"Total NaNs in DataFrame: {total_nans}")

In [ ]:
# ------------------------------------------------------------------
# 2.  Numeric coercion, per-area, per-capita
# ------------------------------------------------------------------
# --- after `stl = stl_tr.merge(acs, on="geoid", how="left")` ---

# 1) Make sure your two denominators are numeric, coercing errors to NaN
# stl['aland']                = pd.to_numeric(stl['aland'],                errors='coerce')
# stl['S0101_C01_001E']       = pd.to_numeric(stl['S0101_C01_001E'],       errors='coerce')
stl = data.copy()
# 2) Build “safe” denominators (zeros → 1)
aland_safe = stl['aland'].replace(0, 1)
pop_safe   = stl['S0101_C01_001E'].replace(0, 1)

# 3) Exclude columns you never want to transform
never = {
    'geoid',
    'aland',
    'S0101_C01_001E'
}

# 4) Identify candidate columns (everything else)
candidates = [c for c in stl.columns if c not in never]

# # 5) Select only those that are fully numeric
# fully_numeric = []
# for col in candidates:
#     conv = pd.to_numeric(stl[col], errors='coerce')
#     # if *every* row converts cleanly, keep it
#     if not conv.isna().any():
#         fully_numeric.append(col)
#         stl[col] = conv  # overwrite with numeric version

# 6) Create per‐area and per‐capita columns just as you did in training
for col in candidates:
    stl[f'{col}_per_ALAND']  = stl[col] / aland_safe
    stl[f'{col}_per_capita'] = stl[col] / pop_safe

# ------------------------------------------------------------------
# 3.  Z-score all numeric cols *within* STL
# ------------------------------------------------------------------

feature_cols = [col for col in stl.columns if col != 'geoid'] 
def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / (sd if sd > 0 else 1)

stl_z = stl.copy()
stl_z[feature_cols] = stl[feature_cols].apply(zscore)

# ------------------------------------------------------------------
# 4.  5-NN spatial lags on exactly the *same* base lists
# ------------------------------------------------------------------
coords_stl = stl_z[['centroid_x','centroid_y']].values
nbrs  = NearestNeighbors(n_neighbors=6,algorithm='ball_tree').fit(coords_stl)
_, idxs_stl = nbrs.kneighbors(coords_stl)

def compute_lag_cols(base_cols, df, idxs_stl, suffix="_lag"):
    lag_cols=[]
    for c in base_cols:
        if c not in df: continue
        name=f"{c}{suffix}"
        if name not in df:
            arr = df[c].values
            df[name] = np.array([arr[idx[1:]].mean() for idx in idxs_stl])
        lag_cols.append(name)
    return lag_cols

lags  = compute_lag_cols(union_feats,  stl_z, idxs_stl)


In [ ]:

# ------------------------------------------------------------------
# 5.  Predict on STL: pipelines inside stack pick their columns 
# ------------------------------------------------------------------
# since each sub-pipeline uses ColumnTransformer(feat_…), you can
# just pass the whole stl_z to stack.predict:


preds_z   = final_stack_hgb.predict(stl_z)
stl_z['predicted_trips_z']        = preds_z               # in z-units


In [ ]:
stl_z.columns

In [ ]:


# ------------------------------------------------------------------
# 6.  Done – inspect or save
# ------------------------------------------------------------------
stl_z[['geoid','predicted_trips_z']].to_csv("./stl/stl_z_preds_2010.csv",index=False)
print("→ STL predictions ready in stl_z_preds_2010.csv")

In [ ]:
# Old HGB keep/wins correlation tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.linear_model import LassoCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors

# Assumes data_z (z-scored DataFrame with centroid_x, centroid_y) and corrs (feature-city correlations) are defined

# Precompute 5-NN neighbor indices on tract centroids
coords = data_z[['centroid_x', 'centroid_y']].values
nbrs_model = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs_model.kneighbors(coords)

groups = data_z['city'].values
outer_cv = GroupKFold(n_splits=len(np.unique(groups)))

results = []
best_score = -np.inf
best_config = None
y = data_z['trips_per_day'].values

for keep in [0.10, 0.15, 0.20, 0.25, 0.30]:
    for wins in [5, 6, 7, 8, 9, 10]:
        print(f"\n=== Testing keep_thresh={keep}, wins_thresh={wins} ===")
        # 1) Correlation strength & wins filter
        abs_corrs = corrs.abs()
        summary = (
            abs_corrs
              .agg(['mean','median'], axis=1)
              .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
              .assign(wins=(abs_corrs >= keep).sum(axis=1))
        )
        selected = summary.query("mean_abs_r >= @keep and wins >= @wins").index.tolist()
        print(f"After corr filter: {len(selected)} features")
        if len(selected) < 5:
            continue

        # 2) Drop highly collinear pairs (|r| > 0.90)
        upper = (
            data_z[selected]
              .corr()
              .abs()
              .where(np.triu(np.ones((len(selected), len(selected)), dtype=bool), k=1))
        )
        to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
        selected2 = [c for c in selected if c not in to_drop]
        print(f"After collinearity filter: {len(selected2)} features")
        if len(selected2) < 5:
            continue
        
                # 3) Same-sign across cities filter
        corrs_sel = corrs.loc[selected2]
        signs_sel = corrs_sel.apply(np.sign)
        modal_sign = signs_sel.mode(axis=1)[0]
        sign_frac = (signs_sel.eq(modal_sign, axis=0).sum(axis=1) / signs_sel.shape[1])
        selected2 = sign_frac[sign_frac == 1.0].index.tolist()
        print(f"After same-sign filter: {len(selected2)} features")


        # 3) LassoCV filter (drop zero coefficients)
        X_lasso = data_z[selected2].values
        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_lasso, y)
        coefs = pd.Series(lasso.coef_, index=selected2)
        selected3 = coefs[coefs != 0].index.tolist()
        print(f"After LassoCV filter: {len(selected3)} features")
        if len(selected3) < 5:
            continue

        # 4) Build spatial lag features for the Lasso-selected set
        X_base = data_z[selected3].values
        lag_arrays = []
        for col in selected3:
            arr = data_z[col].values
            lag = np.array([arr[neighbors[1:]].mean() for neighbors in idxs])
            lag_arrays.append(lag)
        X_lag = np.column_stack(lag_arrays)
        X_hgb = np.hstack([X_base, X_lag])

        # 5) Evaluate HGB LOCO R² with lag features
        hgb = HistGradientBoostingRegressor(
            learning_rate=0.05, max_depth=None,
            l2_regularization=0.0, max_iter=600, random_state=42
        )
        scores = cross_val_score(
            hgb, X_hgb, y,
            cv=outer_cv.split(data_z, groups=groups),
            scoring='r2', n_jobs=-1
        )
        mean_r2 = scores.mean()
        print(f"HGB LOCO R² with lags: {mean_r2:.3f}")

        # Record results
        results.append({
            'keep_thresh': keep,
            'wins_thresh': wins,
            'n_corr_feats': len(selected),
            'n_collinear_removed': len(to_drop),
            'n_lasso_feats': len(selected3),
            'r2_with_lags': mean_r2
        })
        if mean_r2 > best_score:
            best_score = mean_r2
            best_config = results[-1]

# Summarize best configuration
print("\n=== Best configuration for HGB ===")
print(best_config)
print("\nFull grid results:")
pd.DataFrame(results)



In [ ]:
# Old elasticnet parameter tuning
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ElasticNet
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.neighbors import NearestNeighbors

# — your “best” thresholds for ElasticNet:
KEEP_THRESH, WINS_THRESH = 0.10, 7
REDUNDANCY_R = 0.90

# 1) Precompute 5-NN on centroids
coords = data_z[['centroid_x','centroid_y']].values
nbrs = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs.kneighbors(coords)

# 2) Correlation + wins filter
abs_corrs = corrs.abs()
summary = (
    abs_corrs
      .agg(['mean','median'], axis=1)
      .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
      .assign(wins=(abs_corrs >= KEEP_THRESH).sum(axis=1))
)
sel = summary.query("mean_abs_r >= @KEEP_THRESH and wins >= @WINS_THRESH").index.tolist()

# 3) Drop highly collinear pairs
upper = data_z[sel].corr().abs().where(
    np.triu(np.ones((len(sel),len(sel)),dtype=bool), k=1)
)
to_drop = [c for c in upper.columns if any(upper[c] > REDUNDANCY_R)]
sel = [c for c in sel if c not in to_drop]

# 4) Same-sign consistency filter
corrs_sel = corrs.loc[sel]
signs    = corrs_sel.apply(np.sign)
modal    = signs.mode(axis=1)[0]
sign_frac= signs.eq(modal,axis=0).sum(axis=1)/signs.shape[1]
sel = sign_frac[sign_frac == 1.0].index.tolist()

# 5) LassoCV pruning
X_las = data_z[sel].values
y     = data_z['trips_per_day'].values
lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_las, y)
coefs = np.abs(lasso.coef_)
sel4  = [feat for feat, coef in zip(sel, coefs) if coef != 0]
print(f"Kept {len(sel4)} features after LassoCV")

# 6) Build spatial lag features on sel4
X_base     = data_z[sel4].values
lag_arrays = []
for col in sel4:
    arr = data_z[col].values
    lag = np.array([ arr[nbrs_idx[1:]].mean() for nbrs_idx in idxs ])
    lag_arrays.append(lag)
X_full = np.hstack([X_base, np.column_stack(lag_arrays)])
print(f"Final feature matrix shape: {X_full.shape}")

# 7) LOCO GridSearchCV for ElasticNet (expanded grid)
groups = data_z['city'].values
outer  = GroupKFold(n_splits=len(np.unique(groups)))

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('enet', ElasticNet(random_state=42, warm_start=True))
])

param_grid = {
    # 10 values from 1e-4 → 1e2
    'enet__alpha':     np.logspace(-4, 2, 10),
    # 6 l1_ratios from mostly ridge → pure lasso
    'enet__l1_ratio':  [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    # 3 convergence tolerances
    'enet__tol':       [1e-6, 1e-5, 1e-4],
    # 3 max_iter settings
    'enet__max_iter':  [1000, 3000, 5000],
    # whether to force non-negative coefficients
    'enet__positive':  [False, True]
}

search = GridSearchCV(
    pipe,
    param_grid,
    cv=outer.split(X_full, y, groups=groups),
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

search.fit(X_full, y)

print("ElasticNet best params:", search.best_params_)
print("ElasticNet best LOCO R²:", search.best_score_)



In [ ]:
# Old RF parameter tuning
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.neighbors import NearestNeighbors

# — best thresholds for RF:
KEEP_THRESH, WINS_THRESH = ?
REDUNDANCY_R = 0.90

# Precompute neighbors
coords = data_z[['centroid_x','centroid_y']].values
nbrs = NearestNeighbors(n_neighbors=6).fit(coords)
_, idxs = nbrs.kneighbors(coords)

# 1) corr + wins
abs_corrs = corrs.abs()
summary = (
    abs_corrs
      .agg(['mean','median'],axis=1)
      .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
      .assign(wins=(abs_corrs >= KEEP_THRESH).sum(axis=1))
)
sel = summary.query("mean_abs_r>=@KEEP_THRESH and wins>=@WINS_THRESH").index.tolist()

# 2) drop collinear
upper = data_z[sel].corr().abs().where(
    np.triu(np.ones((len(sel),len(sel)),dtype=bool),k=1)
)
to_drop = [c for c in upper.columns if any(upper[c]>REDUNDANCY_R)]
sel = [c for c in sel if c not in to_drop]

# 3) same-sign
corrs_sel = corrs.loc[sel]
signs = corrs_sel.apply(np.sign)
modal = signs.mode(axis=1)[0]
sign_frac = signs.eq(modal,axis=0).sum(axis=1)/signs.shape[1]
sel = sign_frac[sign_frac==1.0].index.tolist()

# 4) LassoCV prune
X_las = data_z[sel].values
y = data_z['trips_per_day'].values
lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_las,y)
coefs = pd.Series(lasso.coef_, index=sel)
sel = coefs[coefs!=0].index.tolist()

# 5) spatial lags
X_base = data_z[sel].values
lags = [ np.array([ data_z[c].values[n[1:]].mean() for n in idxs ]) 
         for c in sel ]
X_full = np.hstack([X_base, np.column_stack(lags)])

# 6) LOCO CV hyperparam search
groups = data_z['city'].values
outer = GroupKFold(n_splits=len(np.unique(groups)))

rf = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators':   [200, 400, 800],
    'max_depth':      [None, 10, 20],
    'min_samples_leaf':[1, 3, 5]
}

search = GridSearchCV(
    rf, param_grid,
    cv=outer.split(X_full, y, groups=groups),
    scoring='r2', n_jobs=-1
)
search.fit(X_full, y)

print("RF best params:", search.best_params_)
print("RF best LOCO R²:", search.best_score_)


In [ ]:
# Old HGB parameter tuning
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.neighbors import NearestNeighbors

# — best thresholds for HGB:
KEEP_THRESH, WINS_THRESH = 0.15, 7
REDUNDANCY_R = 0.90

# Precompute neighbors
coords = data_z[['centroid_x','centroid_y']].values
nbrs = NearestNeighbors(n_neighbors=6).fit(coords)
_, idxs = nbrs.kneighbors(coords)

# 1) corr + wins
abs_corrs = corrs.abs()
summary = (
    abs_corrs
      .agg(['mean','median'],axis=1)
      .rename(columns={'mean':'mean_abs_r','median':'median_abs_r'})
      .assign(wins=(abs_corrs >= KEEP_THRESH).sum(axis=1))
)
sel = summary.query("mean_abs_r>=@KEEP_THRESH and wins>=@WINS_THRESH").index.tolist()

# 2) drop collinear
upper = data_z[sel].corr().abs().where(
    np.triu(np.ones((len(sel),len(sel)),dtype=bool),k=1)
)
to_drop = [c for c in upper.columns if any(upper[c]>REDUNDANCY_R)]
sel = [c for c in sel if c not in to_drop]

# 3) same-sign
corrs_sel = corrs.loc[sel]
signs = corrs_sel.apply(np.sign)
modal = signs.mode(axis=1)[0]
sign_frac = signs.eq(modal,axis=0).sum(axis=1)/signs.shape[1]
sel = sign_frac[sign_frac==1.0].index.tolist()

# 4) LassoCV prune 
X_las = data_z[sel].values
y = data_z['trips_per_day'].values
lasso = LassoCV(cv=5, random_state=42, n_jobs=-1).fit(X_las,y)
coefs = pd.Series(lasso.coef_, index=sel)
sel = coefs[coefs!=0].index.tolist()

# sel4 = list of columns that survived your LassoCV filter
X_las = data_z[sel4].values


# …then proceed with your LOCO‐CV / GridSearchCV on X_model…



# # 5) spatial lags
X_base = data_z[sel].values
lags = [ np.array([ data_z[c].values[n[1:]].mean() for n in idxs ]) 
         for c in sel ]
X_full = np.hstack([X_base, np.column_stack(lags)])

# 6) HGB LOCO hyperparam tuning
groups = data_z['city'].values
outer = GroupKFold(n_splits=len(np.unique(groups)))

#hgb = HistGradientBoostingRegressor(random_state=42)

hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,       # you can still grid-search over this
    max_depth=None,           # likewise
    min_samples_leaf=50,      # likewise
    l2_regularization=0.1,    # likewise
    max_iter=2000,            # up to 2,000 boosting rounds
    early_stopping=True,      # stop when validation loss plateaus
    validation_fraction=0.1,  # hold out 10% of each CV train fold
    n_iter_no_change=50,      # stop after 50 rounds with no improvement
    tol=1e-4,                 # minimum loss improvement to reset counter
    random_state=42
)
param_grid = {
    'learning_rate':     [0.01, 0.05, 0.1],
    'max_depth':         [None, 5, 10],
    'min_samples_leaf':  [20, 50, 100],
    'l2_regularization':[0.0, 0.1, 1.0]
}

search = GridSearchCV(
    hgb, param_grid,
    cv=outer.split(X_full, y, groups=groups),
    scoring='r2', n_jobs=-1
)
search.fit(X_full, y)

print("HGB best params:", search.best_params_)
print("HGB best LOCO R²:", search.best_score_)


In [ ]:
best_hgb = search.best_estimator_
print(best_hgb)
print(best_hgb.get_params())


In [ ]:
print("tuning X_full shape:", X_full.shape)
print("pipeline X shape:   ", X.shape)


In [ ]:
# ------------------------------------------------------------------
# 0.  Define a lag-only helper that never mutates your base list
# ------------------------------------------------------------------
def compute_lag_cols(base_cols, df, idxs, suffix="_lag5"):
    lag_cols = []
    for col in base_cols:
        lag_name = f"{col}{suffix}"
        # If the column doesn’t yet exist, build it
        if lag_name not in df.columns:
            arr = df[col].values
            df[lag_name] = np.array([
                arr[nbrs_idx[1:]].mean() for nbrs_idx in idxs
            ])
        lag_cols.append(lag_name)
    return lag_cols



In [ ]:
from sklearn.compose   import ColumnTransformer
from sklearn.pipeline  import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble  import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet, RidgeCV
from sklearn.model_selection import GroupKFold, cross_val_score

coords = data_z[['centroid_x','centroid_y']].values
nbrs   = NearestNeighbors(n_neighbors=6, algorithm='ball_tree').fit(coords)
_, idxs = nbrs.kneighbors(coords)      # idxs[i] = [i, nn1, nn2, nn3, nn4, nn5]

# # 2) Compute lag columns from the base feature lists:
enet_df = pd.read_csv('./correlation_results_2/features/enet_2010.csv')
rf_df = pd.read_csv('./correlation_results_2/features/rf_2010.csv')
hgb_df = pd.read_csv('./correlation_results_2/features/hgb_2010.csv')

base_enet = enet_df['features'].tolist()
base_rf = rf_df['features'].tolist()
base_hgb = hgb_df['features'].tolist()

banned_feats = ['S0101_C02_001E', 'S0802_C03_004E', 'S0802_C03_096E','S0802_C03_038E']
base_enet = [feat for feat in base_enet if feat not in banned_feats]
base_rf = [feat for feat in base_rf if feat not in banned_feats]
base_hgb = [feat for feat in base_hgb if feat not in banned_feats]


lag_enet  = compute_lag_cols(base_enet, data_z, idxs)
lag_rf    = compute_lag_cols(base_rf,   data_z, idxs)
lag_hgb   = compute_lag_cols(base_hgb,  data_z, idxs)

feat_enet = base_enet + lag_enet
feat_rf   = base_rf   + lag_rf
feat_hgb  = base_hgb  + lag_hgb
# (optionally each has its own spatial-lag columns appended)

# ------------------------------------------------------------------
# 2. Build a pipeline *per model*
# ------------------------------------------------------------------
def make_pipe(cols, model, scale=False):
    """Return a Pipeline that picks `cols`, scales if asked, then fits `model`."""
    selector = ColumnTransformer(
        [("keep", "passthrough", cols)], remainder='drop'
    )
    steps = [("select", selector)]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

enet_pipe = make_pipe(
    feat_enet,
    #ElasticNet(alpha=0.01, l1_ratio=0.7, max_iter=1000, random_state=42),
    ElasticNet(
        alpha     = 0.046415888336127774,
        l1_ratio  = 0.2,
        max_iter  = 1000,
        tol       = 1e-05,
        positive  = False,
        random_state=42
    ),
    scale=True

)

rf_pipe   = make_pipe(
    feat_rf,
    RandomForestRegressor(
        n_estimators=400, max_depth=None, min_samples_leaf=5,
        random_state=42
    )
)
hgb_pipe  = make_pipe(
    feat_hgb,
     HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=10,
    min_samples_leaf=20,
    l2_regularization=1.0,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50,
    tol=1e-4,
    random_state=42
))

# ------------------------------------------------------------------
# 3. Stack them
# ------------------------------------------------------------------
stack = StackingRegressor(
    estimators=[
        ("enet", enet_pipe)
        ,("rf"  , rf_pipe)
        ,("hgb" , hgb_pipe)
    ],
    final_estimator=ElasticNet(alpha=0.01, l1_ratio=0.1, random_state=42),
    passthrough=False,      # final_estimator sees ONLY the three predictions
    n_jobs=-1
)

# ------------------------------------------------------------------
# 4. Evaluate with LOCO CV
# ------------------------------------------------------------------
X = data_z        # the full DataFrame; each ColumnTransformer will pick its own cols
y = data_z['trips_per_day'].values
groups = data_z['city'].values
loco_cv = GroupKFold(n_splits=data_z['city'].nunique())

scores = cross_val_score(
    stack, X, y,
    cv=loco_cv.split(X, y, groups=groups),
    scoring='r2',
    n_jobs=-1
)
print(f"Stacked model LOCO R² = {scores.mean():.3f}  (+/- {scores.std():.3f})")


In [ ]:
from sklearn.metrics import r2_score
from sklearn.base import clone
import pandas as pd

# --------------------------------------------------------
# 1 .  Build once — it’s expensive to refit inside the loop
# --------------------------------------------------------
stack_clone = clone(stack)                     # keep original untouched

city_scores = {}                               # {city : R²}

for train_idx, test_idx in loco_cv.split(X, y, groups=groups):
    city_name = data_z['city'].iloc[test_idx[0]]   # every row in this fold
                                                   # belongs to the same city
    # --- fit on the 10 remaining cities
    stack_clone.fit(X.iloc[train_idx], y[train_idx])

    # --- predict that held-out city
    y_pred = stack_clone.predict(X.iloc[test_idx])

    city_scores[city_name] = r2_score(y[test_idx], y_pred)

# Pretty print
city_r2 = pd.Series(city_scores).sort_values(ascending=False)
print("\nLOCO R² by city")
print(city_r2.to_string(float_format="%.3f"))

print(f"\nMean R²  : {city_r2.mean():.3f}")
print(f"Std-dev R²: {city_r2.std():.3f}")


In [ ]:
# ------------------------------------------------------------------
# 5.  Which variables actually drive the ≈0.437 LOCO score?
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance

# ----------------------------------------------------
# (A) Internal importance from each fitted base model
# ----------------------------------------------------
from sklearn.inspection import permutation_importance

def _kept_cols(pipe):
    """Column names that survive the ColumnTransformer inside *pipe*."""
    return pipe.named_steps["select"].transformers_[0][2]

def _model_importance(pipe, cols, X_fit, y_fit):
    """Return a pd.Series of importances, one per *cols*."""
    mdl = pipe.named_steps["model"]

    # 1) Linear models with coefficients
    if hasattr(mdl, "coef_"):
        scaler = pipe.named_steps.get("scale")
        if scaler is not None:
            imp = np.abs(mdl.coef_ / scaler.scale_)
        else:
            imp = np.abs(mdl.coef_)

    # 2) Tree ensembles exposing impurity-based importances
    elif hasattr(mdl, "feature_importances_"):
        imp = mdl.feature_importances_

    # 3) Fallback: permutation importance on the fitted pipeline itself
    else:
        r = permutation_importance(
                pipe, X_fit[cols], y_fit,
                n_repeats=5, random_state=42, scoring="r2", n_jobs=-1
        )
        imp = r.importances_mean

    return pd.Series(imp / imp.sum(), index=cols)     # normalise

# ---------- run it -------------------------------------------------
stack.fit(X, y)                                       # one global fit

agg_imp = pd.Series(dtype=float)
for name, subpipe in stack.named_estimators_.items():
    cols = _kept_cols(subpipe)
    agg_imp = agg_imp.add(_model_importance(subpipe, cols, X, y), fill_value=0)

agg_imp /= len(stack.named_estimators_)               # equal weight per model

print("\n=== Top variables (internal importance, averaged) ===")
print(agg_imp.sort_values(ascending=False).head(25))
# ----------------------------------------------------
# (B) Out-of-sample permutation importance (LOCO CV)
#     – slower but leakage-free
# ----------------------------------------------------
perm_scores = []
for train_idx, test_idx in loco_cv.split(X, y, groups):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    stack.fit(X_tr, y_tr)                           # fit on k-1 cities
    r = permutation_importance(
            stack, X_te, y_te,
            scoring="r2",
            n_repeats=10,
            random_state=42,
            n_jobs=-1
    )
    perm_scores.append(pd.Series(r.importances_mean, index=X.columns))

perm_imp = pd.concat(perm_scores, axis=1).mean(axis=1)
print("\n=== Top variables (permutation importance, LOCO-averaged) ===")
print(perm_imp.sort_values(ascending=False).head(25))


In [ ]:
print(perm_imp.sort_values(ascending=False).head(25))


In [ ]:

# ------------------------------------------------------------------
# 1.  Load St Louis tracts + ACS exactly as in your train prep
# ------------------------------------------------------------------
from functools import reduce

# 1) Your STL tract geometry + minimal cols
mo_gdf = gpd.read_file("./tract_shapefiles/tl_2023_mo_tract/tl_2023_29_tract.shp")
stl_tr = mo_gdf.query("COUNTYFP in ['510','189']").copy()
stl_tr['geoid']      = stl_tr['GEOID'].str[-11:]
stl_tr['centroid_x'] = stl_tr.geometry.centroid.x
stl_tr['centroid_y'] = stl_tr.geometry.centroid.y
stl_tr = stl_tr.rename(columns={'ALAND':'aland'})[['geoid','aland','centroid_x','centroid_y']]

# 2) Load & keep *all* estimate columns (E), drop the M ones

census_files = [
    "./census_data/S0101_2023.csv",
    "./census_data/S0601_2023.csv",
    "./census_data/S0701_2023.csv",
    "./census_data/S0801_2023.csv",
    "./census_data/S0802_2020.csv",
    "./census_data/DP03_2023.csv",
]
frames = []
for path in census_files:
    df = pd.read_csv(path, dtype={"GEOID": str})
    df = df.rename(columns={"GEOID": "geoid", "GEO_ID": "geoid"})
    df["geoid"] = df["geoid"].str[-11:]
    # pick only columns that end with "E", *except* the tract name
    est_cols = [c for c in df.columns 
                if c.endswith("E") 
                and c != "NAME"         # <-- drop the NAME field
                and not c.endswith("M")] # also skip any margin‐of‐error
    frames.append(df[["geoid", *est_cols]])

# 3) Outer‐merge so we get one wide ACS frame
acs = reduce(lambda L, R: L.merge(R, on="geoid", how="outer"), frames)

# 4) Join onto your STL tracts
stl = stl_tr.merge(acs, on="geoid", how="left")

# quick check
print("Got population column?", "S0101_C01_001E" in stl.columns)
print("Nulls in population:", stl["S0101_C01_001E"].isna().sum())



In [ ]:
stl.columns

In [ ]:
stl.shape

In [ ]:
print("Original shape: " , stl.shape)
data = stl.dropna(how='all') #thresh = require that many non na values to not be dropped
print("After removing rows missing every entry: " , data.shape)
data = data.dropna(how='all', axis=1) #thresh = how many non na values to not be dropped
print("After removing cols missing every entry: " , data.shape)

In [ ]:
exclude = ['geoid']
# all other columns: attempt to convert to numeric
for col in data.columns:
    if col not in exclude:
        data[col] = pd.to_numeric(data[col], errors='coerce')
print(data.shape)
data = data.dropna(axis=1, how='all')
print(data.shape)

for col in data.columns:
    if col == 'geoid':
        continue
    median = data[col].median()
    data.loc[:, col] = data[col].fillna(median)
print(data.shape)
data = data.dropna(axis=1, how='any')
print(data.shape)
total_nans = data.isna().sum().sum()
print(f"Total NaNs in DataFrame: {total_nans}")

In [ ]:
features_in_stl = pd.DataFrame(data.columns, columns=['features'])
features_in_stl.to_csv('./stl/features_in_stl.csv')

In [ ]:

# ------------------------------------------------------------------
# 2.  Numeric coercion, per-area, per-capita
# ------------------------------------------------------------------
# --- after `stl = stl_tr.merge(acs, on="geoid", how="left")` ---

# 1) Make sure your two denominators are numeric, coercing errors to NaN
# stl['aland']                = pd.to_numeric(stl['aland'],                errors='coerce')
# stl['S0101_C01_001E']       = pd.to_numeric(stl['S0101_C01_001E'],       errors='coerce')
stl = data.copy()
# 2) Build “safe” denominators (zeros → 1)
aland_safe = stl['aland'].replace(0, 1)
pop_safe   = stl['S0101_C01_001E'].replace(0, 1)

# 3) Exclude columns you never want to transform
never = {
    'geoid',
    'aland',
    'S0101_C01_001E'
}

# 4) Identify candidate columns (everything else)
candidates = [c for c in stl.columns if c not in never]

# # 5) Select only those that are fully numeric
# fully_numeric = []
# for col in candidates:
#     conv = pd.to_numeric(stl[col], errors='coerce')
#     # if *every* row converts cleanly, keep it
#     if not conv.isna().any():
#         fully_numeric.append(col)
#         stl[col] = conv  # overwrite with numeric version

# 6) Create per‐area and per‐capita columns just as you did in training
for col in candidates:
    stl[f'{col}_per_ALAND']  = stl[col] / aland_safe
    stl[f'{col}_per_capita'] = stl[col] / pop_safe

# ------------------------------------------------------------------
# 3.  Z-score all numeric cols *within* STL
# ------------------------------------------------------------------

feature_cols = [col for col in stl.columns if col != 'geoid'] 
def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / (sd if sd > 0 else 1)

stl_z = stl.copy()
stl_z[feature_cols] = stl[feature_cols].apply(zscore)

# ------------------------------------------------------------------
# 4.  5-NN spatial lags on exactly the *same* base lists
# ------------------------------------------------------------------
coords = stl_z[['centroid_x','centroid_y']].values
nbrs  = NearestNeighbors(n_neighbors=6,algorithm='ball_tree').fit(coords)
_, idxs = nbrs.kneighbors(coords)

def compute_lag_cols(base_cols, df, idxs, suffix="_lag5"):
    lag_cols=[]
    for c in base_cols:
        if c not in df: continue
        name=f"{c}{suffix}"
        if name not in df:
            arr = df[c].values
            df[name] = np.array([arr[idx[1:]].mean() for idx in idxs])
        lag_cols.append(name)
    return lag_cols

lag_enet = compute_lag_cols(feat_enet, stl_z, idxs)
lag_rf   = compute_lag_cols(feat_rf,   stl_z, idxs)
lag_hgb  = compute_lag_cols(feat_hgb,  stl_z, idxs)


In [ ]:
stl_z.columns

In [ ]:

# ------------------------------------------------------------------
# 5.  Predict on STL: pipelines inside stack pick their columns 
# ------------------------------------------------------------------
# since each sub-pipeline uses ColumnTransformer(feat_…), you can
# just pass the whole stl_z to stack.predict:

y_mean = data_z['trips_per_day'].mean()
y_std  = data_z['trips_per_day'].std(ddof=0)

preds_z   = stack.predict(stl_z)
stl_z['predicted_trips_z']        = preds_z               # in z-units
stl_z['predicted_trips_per_day'] = preds_z * y_std + y_mean


# ------------------------------------------------------------------
# 6.  Done – inspect or save
# ------------------------------------------------------------------
stl_z[['geoid','predicted_trips_z', 'predicted_trips_per_day']].to_csv("stl_z_preds_2010.csv",index=False)
print("→ STL predictions ready in stl_z_preds_2010.csv")


In [ ]:
y_mean